# 第 69 课｜真实小模型 LoRA 微调前预检

**训练数据边界、基座冻结、adapter 身份、梯度连通与显存预算。**

第67课建立 Tokenizer 与 Chat Template，第68课审计本地 checkpoint 的只读推理身份。本课继续问：**在按下“训练”之前，怎样发现错分词、错误 loss mask、基座误解冻、目标模块错误、adapter接错基座和显存预算误判？**

默认离线执行真正的两层因果 Transformer 和手写 Q/V LoRA，做真实的前向、反向和参数更新；但**基座是随机初始化的教学模型，不是Qwen或任何预训练LLM**。本课的 loss 是 teacher-forced SFT NLL，不是自由生成准确率。所有数据由纯代码合成。

可选检查用户已有的**受信任本地 HF Safetensors**；再进一步的真实 HF+PEFT 单步梯度烟雾测试必须显式请求，依赖不会自动安装。没有实际用户模型时，不会假装它已经测试过。

## 0. 运行顺序与边界

从已有 PyTorch 的 Python 3 内核中**重启并顺序运行全部单元**。默认完成固定检查、离线 toy LoRA 训练及独立磁盘重载；CPU 足够，不依赖 RTX 3060。

只想先看机制时，可将后面的 `RUN_FULL_TOY` 改为 `False`。所有练习开关默认 `False`。完整脚本中也有单独的 `--checks-only`、`--inspect-local`、`--local-peft-smoke` 和 `--reload-run` 入口。

文件是独立讲义，不从第68课导入任何代码。实验每次写入新时间戳目录；绝不覆盖模型源目录。本课不会下载模型、上传权重、执行远端自定义代码或自动 pip 安装。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第69课：LoRA微调前预检——训练数据边界、冻结、adapter身份、梯度与资源安全。

默认全离线，使用自建chat tokenizer和随机因果Transformer做真实前向/反向；
可选只读静态检查受信任的本地HF Safetensors；更进一步的本地HF+PEFT单步训练
必须显式请求，且需要用户先行安装依赖，绝不下载或执行远程自定义代码。
没有声称预训练模型训练或3060 CUDA实测。
"""
from __future__ import annotations

import argparse
import copy
import hashlib
import json
import math
import os
import platform
import random
import struct
import sys
import tempfile
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit('需要现有PyTorch环境；本课不自动安装依赖。') from exc

VERSION = 'lesson69-lora-preflight-v1'
IGNORE = -100
SOURCES = {
    'chat_template': 'https://huggingface.co/docs/transformers/main/chat_templating',
    'peft_lora': 'https://huggingface.co/docs/peft/quicktour',
    'peft_reference': 'https://huggingface.co/docs/peft/main/package_reference/lora',
    'transformers_offline': 'https://huggingface.co/docs/transformers/installation',
    'cuda_memory_allocated': 'https://docs.pytorch.org/docs/stable/generated/torch.cuda.memory.max_memory_allocated.html',
    'cuda_memory_reserved': 'https://docs.pytorch.org/docs/stable/generated/torch.cuda.memory.max_memory_reserved.html',
    'pytorch_repro': 'https://docs.pytorch.org/docs/stable/notes/randomness.html',
    'safetensors': 'https://huggingface.co/docs/safetensors/index',
}

class LessonError(ValueError):
    """拒绝含糊的token边界、停止状态、报告格式或资源预算。"""

def require(ok: bool, msg: str) -> None:
    if not bool(ok):
        raise LessonError(msg)

def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True,
                               allow_nan=False) + "\n", encoding="utf-8")

def bounded_str(value: Any, length: int, label: str) -> str:
    require(type(value) is str and len(value.encode("utf-8")) <= length,
            label + "必须是有界字符串")
    return value

def strict_json(text: str):
    bounded_str(text, 8192, "JSON")
    def pairs(items):
        result = {}
        for key, value in items:
            require(key not in result, "重复JSON字段")
            result[key] = value
        return result
    def bad(_):
        raise LessonError("不接受NaN/Infinity")
    try:
        return json.loads(text, object_pairs_hook=pairs, parse_constant=bad)
    except (json.JSONDecodeError, RecursionError, UnicodeError) as exc:
        raise LessonError("JSON无法严格解析") from exc

def device_from_name(name: str):
    require(name in {"cpu", "cuda", "auto"}, "device应为cpu/cuda/auto")
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    require(name != "cuda" or torch.cuda.is_available(), "CUDA不可用，不静默退回CPU")
    return torch.device(name)

def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def state_hash(state: dict[str,torch.Tensor]) -> str:
    h=hashlib.sha256()
    for key,tensor in sorted(state.items()):
        t=tensor.detach().cpu().contiguous()
        h.update(canonical([key,str(t.dtype),list(t.shape)]).encode())
        h.update(t.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

In [2]:
torch.set_num_threads(1)
DEVICE = 'cpu'         # 有可用CUDA且主动选择时可设为cuda；本制作环境没有CUDA实测。
RUN_FULL_TOY = True
OUTPUT_ROOT = Path('lesson_69_outputs')
seed_all(6901)
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__, 'CUDA可用:', torch.cuda.is_available())
print('设备:', DEVICE)

Python: 3.13.5
PyTorch: 2.10.0+cpu CUDA可用: False
设备: cpu


## 1. Chat Template 不是普通字符串拼接

前课的教学 `ByteMergeChatTokenizer` 在本课完整内嵌：结构符和角色各有独立 ID，用户输入中写出的字面 `<im_start>` 仍是普通UTF-8文本，不会变成真实消息边界。

**训练路径使用两个完整模板结果求前缀长度：**

1. `system + user`，开启 assistant generation prompt；
2. `system + user + assistant`，关闭 generation prompt；
3. 要求第1个 token 列表是第2个的**精确前缀**，再把它们之间的界限作为监督 mask。

没有这个等式，就拒绝继续，不根据字符串长度、字符位置或“猜测的 assistant 头”找损失边界。真实 HF tokenizer 也要单独验证。 [HF Chat Template](https://huggingface.co/docs/transformers/main/chat_templating)

In [3]:
class ByteMergeChatTokenizer:
    """自建教学分词器：控制token独立于用户文本，普通文本按UTF-8字节合并。

    不假装实现HF的FastTokenizer或任意模型Chat Template。固定控制ID，避免把用户
    输入里的字面控制符误当结构token。真实应用必须使用对应模型的官方tokenizer/template。
    """
    pad_token_id = 0
    bos_token_id = 1
    eos_token_id = 2
    im_start_token_id = 3
    im_end_token_id = 4
    role_ids = {"system":5, "user":6, "assistant":7}
    byte_offset = 16
    merges = (b"OK", b" 42", b"42", b"hello", b"world", b"\n", b"answer", b"JSON", b"{\"answer\":")

    def __init__(self):
        self.merge_ids = {piece: self.byte_offset + 256 + i for i,piece in enumerate(self.merges)}
        self.ids_to_merge = {v:k for k,v in self.merge_ids.items()}
        self.vocab_size = self.byte_offset + 256 + len(self.merges)
        self.specials = {0:"<pad>",1:"<bos>",2:"<eos>",3:"<im_start>",
                         4:"<im_end>",5:"<system>",6:"<user>",7:"<assistant>"}

    def encode(self, text: str, *, add_special_tokens: bool=False) -> list[int]:
        bounded_str(text, 4096, "文本")
        raw = text.encode("utf-8")
        result = []
        index = 0
        while index < len(raw):
            matches = [piece for piece in self.merges if raw.startswith(piece, index)]
            if matches:
                piece = max(matches, key=len)
                result.append(self.merge_ids[piece]);index += len(piece)
            else:
                result.append(self.byte_offset + raw[index]);index += 1
        return ([self.bos_token_id] if add_special_tokens else []) + result

    def decode(self, ids: list[int], *, skip_special_tokens: bool=False) -> str:
        require(type(ids) is list, "token IDs必须是列表")
        raw = bytearray()
        for i in ids:
            require(type(i) is int and 0 <= i < self.vocab_size, "token ID越界")
            if i in self.specials:
                if skip_special_tokens:
                    continue
                raise LessonError("正文decode不允许控制token；传入skip_special_tokens=True")
            if i in self.ids_to_merge: raw.extend(self.ids_to_merge[i])
            elif self.byte_offset <= i < self.byte_offset + 256: raw.append(i - self.byte_offset)
            else: raise LessonError("保留ID不是合法正文token")
        try:
            return bytes(raw).decode("utf-8",errors="strict")
        except UnicodeDecodeError as exc:
            raise LessonError("不完整UTF-8回答；不要用replacement字符伪装成功") from exc

    def apply_chat_template(self, messages: list[dict], *, tokenize: bool=True,
                            add_generation_prompt: bool=False):
        require(type(messages) is list and bool(messages), "messages需要非空列表")
        require(type(add_generation_prompt) is bool, "generation prompt必须是bool")
        ids = [self.bos_token_id]
        for j,m in enumerate(messages):
            require(type(m) is dict and set(m)=={"role","content"}, "message字段必须恰好role/content")
            require(m["role"] in self.role_ids and type(m["content"]) is str, "角色或内容非法")
            require(not (j == len(messages)-1 and add_generation_prompt and m["role"]=="assistant"),
                    "若最后已有assistant，请使用续写语义，不要再添加generation prompt")
            ids.extend([self.im_start_token_id,self.role_ids[m["role"]]])
            ids.extend(self.encode(m["content"],add_special_tokens=False))
            ids.append(self.eos_token_id if m["role"]=="assistant" else self.im_end_token_id)
        if add_generation_prompt:
            ids.extend([self.im_start_token_id,self.role_ids["assistant"]])
        if tokenize:
            return ids
        # 仅用于人类阅读，控制符是标记，不要将其当普通文本用encode()重新编码。
        visible = []
        for i in ids:
            if i in self.specials: visible.append(self.specials[i])
            else: visible.append(self.decode([i],skip_special_tokens=True) if
                                 self._token_is_full_utf8(i) else "<byte-part>")
        return "".join(visible)

    def _token_is_full_utf8(self, i):
        if i in self.ids_to_merge:
            try: self.ids_to_merge[i].decode("utf-8");return True
            except UnicodeError: return False
        return self.byte_offset <= i < self.byte_offset + 128

    def contract(self) -> dict:
        return {"type":"toy-byte-merge", "specials":self.specials,
                "merges":[p.hex() for p in self.merges],"vocab":self.vocab_size,
                "prefix":"BOS+message start/role/body+end+assistant header",
                "eos":self.eos_token_id,"pad":self.pad_token_id}

In [4]:
tok = ByteMergeChatTokenizer()
print('词表大小:', tok.vocab_size)
print('用户文本中控制符字样的编码:',tok.encode('<im_start>'))
print('真实消息开始符ID:',tok.im_start_token_id)
print('字面文本不会获得结构ID:',tok.im_start_token_id not in tok.encode('<im_start>'))

词表大小: 281
用户文本中控制符字样的编码: [76, 121, 125, 111, 131, 132, 113, 130, 132, 78]
真实消息开始符ID: 3
字面文本不会获得结构ID: True


## 2. 数据进入训练器前，先按来源分组

固定产生12个来源，每来源4条记录，总共48条。来源先按固定种子划分 **8/2/2**，对应 **32/8/8** 条训练、开发、测试。相同来源不能跨侧；重复完整提示也必须拒绝。

人工复制任务：`copy:05,02`，对应唯一 assistant 正文 `{"answer":"02"}`。系统文本、角色和答案 JSON 是固定协议，不是人类偏好数据或私有资料。规则结构在各侧共享，**不把这个拆分叫跨领域泛化测试**。

预检会拒绝 ID重复、来源交叉、错误答案、字段缺失、划分标签不匹配与静默截断。所有训练/选轮仅允许访问训练侧；开发侧用于监测，最终测试只做报告。

In [5]:
SYSTEM_TEXT = 'Return only JSON with one string field named answer.'


def make_dataset(seed: int=6901) -> dict[str,list[dict]]:
    """12个来源×4条；来源分组8/2/2，不能拿测试来源调学习率。"""
    source_ids=list(range(12))
    random.Random(seed).shuffle(source_ids)
    part={a:('train' if i<8 else 'dev' if i<10 else 'test')
          for i,a in enumerate(source_ids)}
    groups={'train':[], 'dev':[], 'test':[]}
    for a in range(12):
        for b in range(4):
            row={'id':f's{a:02d}b{b:02d}', 'source_id':f's{a:02d}',
                 'split':part[a], 'system':SYSTEM_TEXT,
                 'user':f'copy:{a:02d},{b:02d}',
                 'assistant':canonical({'answer':f'{b:02d}'})}
            groups[part[a]].append(row)
    validate_dataset(groups)
    return groups


def validate_dataset(groups: dict[str,list[dict]]) -> None:
    require(type(groups) is dict and set(groups)=={'train','dev','test'}, '需要train/dev/test')
    seen_ids=set();seen_user=set();seen_sources={}
    for split,rows in groups.items():
        require(type(rows) is list and bool(rows),'划分不能为空')
        for row in rows:
            require(type(row) is dict and set(row)=={
                'id','source_id','split','system','user','assistant'},'记录字段不匹配')
            require(all(type(v) is str and v for v in row.values()),'所有字段必须为非空字符串')
            require(row['split']==split,'记录划分标签与所属集合不一致')
            require(row['id'] not in seen_ids and row['user'] not in seen_user,'ID/prompt重复')
            seen_ids.add(row['id']);seen_user.add(row['user'])
            require(row['system']==SYSTEM_TEXT,'系统指令不是固定版本')
            require(row['source_id']==row['id'][:3], '来源ID不一致')
            require(seen_sources.get(row['source_id'],split)==split,'相同来源跨划分')
            seen_sources[row['source_id']]=split
            prefix=row['user'].split(',')
            require(len(prefix)==2 and prefix[0]==f"copy:{int(row['source_id'][1:]):02d}",
                    '用户提示不符合教学固定格式')
            require(len(prefix[1])==2 and prefix[1].isascii() and prefix[1].isdigit(),
                    '复制源不是两位数字')
            require(row['assistant']==canonical({'answer':prefix[1]}),
                    '参考答案与复制任务规则不符')
    require([len(groups[s]) for s in ('train','dev','test')]==[32,8,8],
            '固定数据分组大小改变，不能直接比较原实验')


def messages_for_row(row: dict) -> list[dict]:
    return [{'role':'system','content':row['system']},
            {'role':'user','content':row['user']}]


def encode_training_row(tok: ByteMergeChatTokenizer,row: dict,
                        max_length: int=160) -> dict:
    require(type(max_length) is int and max_length>0,'长度上限非法')
    prompt=tok.apply_chat_template(messages_for_row(row),tokenize=True,
                                    add_generation_prompt=True)
    all_messages=messages_for_row(row)+[{'role':'assistant','content':row['assistant']}]
    full=tok.apply_chat_template(all_messages,tokenize=True,
                                add_generation_prompt=False)
    require(len(full)>len(prompt) and full[:len(prompt)]==prompt,
            'Chat Template的生成头/训练序列不具有精确token前缀关系')
    answer_ids=full[len(prompt):]
    require(answer_ids[-1]==tok.eos_token_id,'每条回答必须包含被监督的真实EOS')
    require(tok.eos_token_id not in answer_ids[:-1], '回答中间不允许结构EOS')
    require(all(i not in tok.specials for i in answer_ids[:-1]),'回答正文含结构token')
    require(len(full)<=max_length,'序列超过上限；不静默截断回答/EOS')
    require(len(prompt)>=2 and len(answer_ids)>=2,'提示或回答不完整')
    labels=[IGNORE]*len(prompt)+answer_ids
    return {'input_ids':full,'labels':labels,'prompt_len':len(prompt),
            'answer_len':len(answer_ids),'id':row['id']}


def collate_training_rows(encoded: list[dict],tok: ByteMergeChatTokenizer,
                          device='cpu') -> dict:
    require(type(encoded) is list and bool(encoded),'空批次')
    width=max(len(x['input_ids']) for x in encoded)
    ids=torch.full((len(encoded),width),tok.pad_token_id,dtype=torch.long,device=device)
    labels=torch.full_like(ids,IGNORE)
    mask=torch.zeros_like(ids,dtype=torch.bool)
    for n,row in enumerate(encoded):
        size=len(row['input_ids'])
        require(size>1 and size==len(row['labels']),'输入与标签不对齐')
        require(row['labels'][0]==IGNORE,'BOS不能被本课作为目标')
        require(row['labels'][-1]==tok.eos_token_id,'EOS未被监督')
        ids[n,:size]=torch.tensor(row['input_ids'],device=device)
        labels[n,:size]=torch.tensor(row['labels'],device=device)
        mask[n,:size]=True
    return {'input_ids':ids,'labels':labels,'attention_mask':mask}

In [6]:
data=make_dataset()
validate_dataset(data)
print('记录数:',{k:len(v) for k,v in data.items()})
print('来源数:',{k:len({r['source_id'] for r in v}) for k,v in data.items()})
print('一条训练记录:',data['train'][0])

记录数: {'train': 32, 'dev': 8, 'test': 8}
来源数: {'train': 8, 'dev': 2, 'test': 2}
一条训练记录: {'id': 's00b00', 'source_id': 's00', 'split': 'train', 'system': 'Return only JSON with one string field named answer.', 'user': 'copy:00,00', 'assistant': '{"answer":"00"}'}


## 3. 标签监督范围：提示没有标签，EOS 必须有标签

教学格式是：

```text
[BOS] [system消息] [user消息] [assistant开始头] assistant正文 [EOS]
            ← 全部为-100 →                       ←监督→
```

`labels` 与 `input_ids` 同位。模型只是输出 logits，**在 loss 函数中统一移位一次**。

**注意两种不同的 mask：** `attention_mask` 让提示能被模型读取；监督 `labels=-100` 使提示和 padding 不直接参与目标。EOS 是一个需要模型学会预测的真实 token，计入监督。

不使用 `encode(prompt)+encode(answer)` 作为真实 BPE tokenizer 的边界替代，因为整体 token 化未必与分段编码相同。教学 tokenizer 虽然可控，仍执行完全相同的前缀验证。

In [7]:
entry=encode_training_row(tok,data['train'][0])
print('prompt token数:',entry['prompt_len'],'回答+EOS token数:',entry['answer_len'])
print('序列总长度:',len(entry['input_ids']))
print('提示标签唯一值:',set(entry['labels'][:entry['prompt_len']]))
print('回答尾部ID:',entry['input_ids'][-1], 'EOS ID:',tok.eos_token_id)
print('回答所有标签与原ID同位:',entry['labels'][entry['prompt_len']:]==entry['input_ids'][entry['prompt_len']:])
batch=collate_training_rows([entry,encode_training_row(tok,data['train'][1])],tok)
print('批次形状:',tuple(batch['input_ids'].shape),'有效监督token数:',(batch['labels']!=-100).sum(-1).tolist())

prompt token数: 63 回答+EOS token数: 7
序列总长度: 70
提示标签唯一值: {-100}
回答尾部ID: 2 EOS ID: 2
回答所有标签与原ID同位: True
批次形状: (2, 70) 有效监督token数: [7, 7]


## 4. 本地 Safetensors 静态检查：读取文件头≠加载模型

本课复制并强化第68课的静态检查：本地路径必须真实存在、不是仓库ID；要求 `config.json`、Safetensors 文件、tokenizer 文件证据。拒绝符号链接、越界/破损头、多分片不匹配、pickle-only 和超出字节预算。

默认只计算元数据与文件头摘要，**不会声称已经哈希整个权重或已经运行模型**。用户主动启用 `full_weight_hash=True` 才读取权重文件全部字节生成内容摘要；哈希相等不等于仓库可信。

静态审计不会导入 Transformers，不会安装依赖，也不加载权重。真实权重更要考虑本地目录来源、依赖版本以及模型自定义代码风险；`local_files_only=True` 与 `trust_remote_code=False` 不是通用安全沙箱。[Safetensors](https://huggingface.co/docs/safetensors/index)

In [8]:
def stream_sha256(path: Path, chunk_bytes=2**20) -> str:
    require(type(chunk_bytes) is int and chunk_bytes > 0,"hash分块非法")
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            data=f.read(chunk_bytes)
            if not data:break
            h.update(data)
    return h.hexdigest()

def ensure_local_directory(path: str|Path) -> Path:
    raw=Path(path).expanduser()
    require(not raw.is_symlink(),"不能通过符号链接提供模型目录")
    require(raw.is_dir() and (raw/"config.json").is_file(),
            "必须是包含config.json的现有本地目录，不接受仓库ID或URL")
    folder=raw.resolve(strict=True)
    require(not folder.is_symlink(),"真实目录不能是符号链接")
    return folder

def read_limited_json(path: Path,max_bytes: int=2_000_000):
    require(path.is_file() and not path.is_symlink(),"JSON文件必须是普通文件，不允许符号链接")
    require(0<path.stat().st_size<=max_bytes,"JSON文件为空或超过安全读取上限")
    return strict_json(path.read_text(encoding="utf-8"))

def safetensors_header(path: Path) -> dict:
    """只读前缀，不通过torch.load打开权重；不是对任意恶意文件的安全认证。"""
    require(path.is_file() and not path.is_symlink(),"权重文件不能是符号链接")
    size=path.stat().st_size
    require(size>=16,"safetensors文件太短")
    with path.open("rb") as source:
        first=source.read(8)
        require(len(first)==8,"权重文件缺失header长度")
        n=struct.unpack("<Q",first)[0]
        require(2<=n<=16*2**20 and n+8<=size,"safetensors header非法或过大")
        head=source.read(n)
        require(len(head)==n,"safetensors header截断")
    header=strict_json(head.decode("utf-8")) if n<=8192 else json.loads(head.decode("utf-8"),
        object_pairs_hook=lambda pairs: _no_duplicate_object(pairs),
        parse_constant=lambda x: (_ for _ in ()).throw(LessonError("非法常量")))
    require(type(header) is dict and bool(header),"safetensors需要JSON对象header")
    data_len=size-8-n
    names=[]
    for key,value in header.items():
        if key=="__metadata__":
            require(type(value) is dict,"safetensors metadata需要对象")
            continue
        require(type(key) is str and 0<len(key)<=4096 and type(value) is dict,
                "safetensors tensor条目非法")
        require(type(value.get("dtype")) is str and type(value.get("shape")) is list,
                "safetensors dtype/shape缺失")
        require(all(type(dim) is int and dim>=0 for dim in value["shape"]),"tensor shape非法")
        offsets=value.get("data_offsets")
        require(type(offsets) is list and len(offsets)==2 and
                all(type(x) is int for x in offsets) and 0<=offsets[0]<=offsets[1]<=data_len,
                "tensor data offsets越界")
        names.append(key)
    require(bool(names),"safetensors文件没有tensor")
    return {"tensors":len(names),"header_sha256":hashlib.sha256(head).hexdigest(),
            "header_size":n,"file_size":size,"data_size":data_len,
            "tensor_names_digest":digest(sorted(names)),"_tensor_names":names}

def _no_duplicate_object(items):
    out={}
    for k,v in items:
        require(k not in out,"重复safetensors JSON字段")
        out[k]=v
    return out

def inspect_local_checkpoint(path: str|Path, *, full_weight_hash=False,
                             max_weight_bytes: int=2*1024**3) -> dict:
    """metadata仅辅助检查文件身份；默认并不声称哈希全部权重。"""
    require(type(full_weight_hash) is bool,"full_weight_hash必须是bool")
    require(type(max_weight_bytes) is int and max_weight_bytes>0,"资源上限需要正整数")
    folder=ensure_local_directory(path)
    cfg=read_limited_json(folder/"config.json")
    require(type(cfg) is dict and type(cfg.get("model_type")) is str,
            "config.json缺少model_type；不会猜测模型架构")
    require(cfg.get("is_encoder_decoder",False) is False,"仅支持decoder-only因果LM")
    shards=sorted(folder.glob("*.safetensors"))
    require(bool(shards),"没有safetensors；本课拒绝回退到pickle权重加载")
    require(all(p.is_file() and not p.is_symlink() for p in shards),"权重分片必须是普通文件")
    index_files=sorted(folder.glob("*.safetensors.index.json"))
    require(len(index_files)<=1,"多个safetensors索引无法确认完整权重")
    require(len(shards)==1 or bool(index_files),
            "多个safetensors分片需要明确的weight_map索引")
    if index_files:
        idx=read_limited_json(index_files[0],max_bytes=8*2**20)
        require(type(idx) is dict and type(idx.get("weight_map")) is dict and bool(idx["weight_map"]),
                "权重索引缺少weight_map")
        listed={str(v) for v in idx["weight_map"].values()}
        require(all(type(v) is str and v==Path(v).name and v.endswith(".safetensors") for v in listed),
                "索引权重文件不能穿越目录")
        require(listed=={p.name for p in shards},"索引和权重分片集合不一致")
    total=sum(p.stat().st_size for p in shards)
    require(total<=max_weight_bytes,
            "safetensors总字节超过显式本地资源闸门；没有尝试载入模型")
    infos=[]
    shard_names={}
    for p in shards:
        header=safetensors_header(p)
        shard_names[p.name]=set(header.pop("_tensor_names"))
        infos.append({"file":p.name,"bytes":p.stat().st_size,
                      "mtime_ns":p.stat().st_mtime_ns,"header":header})
    if index_files:
        mapping=idx["weight_map"]
        require(set(mapping)==set().union(*shard_names.values()),
                "weight_map与各safetensors header的tensor集合不匹配")
        require(all(tensor in shard_names[file_name] for tensor,file_name in mapping.items()),
                "weight_map把tensor映射到了错误的shard")
    if full_weight_hash:
        for part,p in zip(infos,shards):part["full_sha256"]=stream_sha256(p)
    small=[]
    for name in ("config.json","tokenizer.json","tokenizer_config.json","special_tokens_map.json",
                 "generation_config.json","chat_template.jinja","tokenizer.model","spiece.model",
                 "sentencepiece.bpe.model","vocab.json","vocab.txt","merges.txt","added_tokens.json",
                 *(p.name for p in index_files)):
        p=folder/name
        if p.exists():
            require(p.is_file() and not p.is_symlink(),"辅助文件不能是符号链接")
            require(p.stat().st_size<=64*2**20,"配置/tokenizer文件超出本课审计上限")
            small.append({"file":name,"bytes":p.stat().st_size,"sha256":stream_sha256(p),
                          "mtime_ns":p.stat().st_mtime_ns})
    require(bool(small) and small[0]["file"]=="config.json","必须审计config.json")
    identity_material={"shards":infos,"small_files":small,"model_type":cfg["model_type"]}
    return {"scope":"local_files_metadata_only" if not full_weight_hash else "local_full_weight_sha256",
            "directory_label":folder.name,"model_type":cfg["model_type"],
            "weight_file_count":len(shards),"weight_total_bytes":total,
            "weight_mib":round(total/2**20,3),
            "max_weight_bytes_gate":max_weight_bytes,
            "metadata_fingerprint":digest(identity_material),
            "full_weight_content_hashed":full_weight_hash,
            "cryptographic_weight_identity":full_weight_hash,
            "weight_files":infos,"small_files":small,
            "warning":"metadata+header fingerprint is not full tensor identity" if not full_weight_hash
                      else "full local weights hashed; still not a trust/security certificate"}

def validate_output_outside_model(model_dir: Path,output_root: str|Path):
    folder=ensure_local_directory(model_dir)
    out=Path(output_root).expanduser().resolve()
    require(out!=folder and folder not in out.parents,
            "报告目录不能位于模型目录内部，避免修改源checkpoint")

In [9]:
with tempfile.TemporaryDirectory() as demo:
    folder=Path(demo)/'mock_checkpoint'
    fake_hf_folder(folder) if 'fake_hf_folder' in globals() else None
    if not folder.exists():
        folder.mkdir()
        write_json(folder/'config.json',{'model_type':'fixture','is_encoder_decoder':False})
        write_json(folder/'tokenizer.json',{'fixture':'metadata_only'})
        payload=b'\x00\x00\x80?'
        header=canonical({'tensor':{'dtype':'F32','shape':[1],'data_offsets':[0,4]}}).encode()
        (folder/'model.safetensors').write_bytes(struct.pack('<Q',len(header))+header+payload)
    result=inspect_local_checkpoint(folder,max_weight_bytes=1024)
    print('文件数:',result['weight_file_count'],'审计范围:',result['scope'])
    print('真的加载模型了吗？',result['scope']=='local_full_model_training')

文件数: 1 审计范围: local_files_metadata_only
真的加载模型了吗？ False


## 5. 真正的微型因果 Transformer：基座固定，LoRA 只更新 Q/V

模型是两个 Transformer Block，隐藏维度32、4头、绝对位置和自回归掩码。全部 dropout 为0。`q_proj`、`k_proj`、`v_proj`、`o_proj` 分开声明，以便准确证明目标层是谁。

LoRA 公式：

\[W' = W + \frac{\alpha}{r} BA\]

A 初始为小随机值，B 全零，所以在尚未训练时 **policy 的 logits 与基座一致**。基座的 embedding、K/O、MLP、LayerNorm、LM head 均冻结。训练器只允许 A/B 进入优化器。[PEFT LoRA初始化与目标层](https://huggingface.co/docs/peft/main/package_reference/lora)

这份**手写 LoRA adapter 文件不是 PEFT 兼容格式**；可选真实 HF+PEFT 的路径才会使用 `save_pretrained()` 保存标准 PEFT adapter。

In [10]:
@dataclass(frozen=True)
class ToyConfig:
    vocab_size: int
    dim: int=32
    heads: int=4
    layers: int=2
    ff_dim: int=64
    max_length: int=160

    def validate(self):
        require(all(type(x) is int and x>0 for x in asdict(self).values()),'配置必须为正整数')
        require(self.dim%self.heads==0 and self.vocab_size>16,
                'dim/head或词表不匹配')


class CausalSelfAttention(nn.Module):
    def __init__(self,cfg: ToyConfig):
        super().__init__()
        d=cfg.dim
        self.heads=cfg.heads
        self.q_proj=nn.Linear(d,d,bias=False)
        self.k_proj=nn.Linear(d,d,bias=False)
        self.v_proj=nn.Linear(d,d,bias=False)
        self.o_proj=nn.Linear(d,d,bias=False)

    def forward(self,x:torch.Tensor,attention_mask:torch.Tensor):
        b,t,d=x.shape;h=self.heads
        def split(layer):
            return layer(x).reshape(b,t,h,d//h).transpose(1,2)
        q,k,v=split(self.q_proj),split(self.k_proj),split(self.v_proj)
        scores=(q@k.transpose(-1,-2))/math.sqrt(d//h)
        lower=torch.ones(t,t,device=x.device,dtype=torch.bool).tril()
        allowed=lower[None,None,:,:]&attention_mask[:,None,None,:]
        weights=torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
        result=(weights@v).transpose(1,2).contiguous().reshape(b,t,d)
        return self.o_proj(result)


class TransformerBlock(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__()
        self.norm1=nn.LayerNorm(cfg.dim)
        self.attn=CausalSelfAttention(cfg)
        self.norm2=nn.LayerNorm(cfg.dim)
        self.ff=nn.Sequential(nn.Linear(cfg.dim,cfg.ff_dim),nn.GELU(),
                              nn.Linear(cfg.ff_dim,cfg.dim))

    def forward(self,x,mask):
        x=x+self.attn(self.norm1(x),mask)
        return x+self.ff(self.norm2(x))


class TinyChatLM(nn.Module):
    def __init__(self,cfg:ToyConfig):
        super().__init__();cfg.validate();self.cfg=cfg
        self.embed=nn.Embedding(cfg.vocab_size,cfg.dim)
        self.position=nn.Embedding(cfg.max_length,cfg.dim)
        self.blocks=nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm=nn.LayerNorm(cfg.dim)
        self.head=nn.Linear(cfg.dim,cfg.vocab_size,bias=False)

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor):
        require(input_ids.ndim==2 and input_ids.dtype==torch.long,'输入必须为[B,T] long')
        b,t=input_ids.shape
        require(b>0 and t>1 and t<=self.cfg.max_length,'形状或上下文超限')
        require(attention_mask.shape==input_ids.shape and attention_mask.dtype==torch.bool,
                'attention mask形状/type不符')
        require(bool(attention_mask[:,0].all()),'不接受左补齐/空提示')
        require(not bool(((~attention_mask[:,:-1])&attention_mask[:,1:]).any()),
                '只允许右补齐')
        positions=torch.arange(t,device=input_ids.device)
        x=self.embed(input_ids)+self.position(positions)[None]
        for layer in self.blocks:x=layer(x,attention_mask)
        return self.head(self.norm(x))


class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,rank=4,alpha=8.0):
        super().__init__()
        require(type(base) is nn.Linear and type(rank) is int and rank>0,
                '手写LoRA仅支持nn.Linear与正整数rank')
        require(type(alpha) in (int,float) and math.isfinite(alpha) and alpha>0,
                'LoRA alpha非法')
        self.base=base
        self.base.requires_grad_(False)
        self.rank=rank;self.alpha=float(alpha);self.enabled=True
        self.A=nn.Parameter(base.weight.new_empty(rank,base.in_features))
        self.B=nn.Parameter(base.weight.new_zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))

    def forward(self,x):
        result=self.base(x)
        if not self.enabled:return result
        return result+(self.alpha/self.rank)*F.linear(F.linear(x,self.A),self.B)


def inject_lora(base:TinyChatLM,rank=4,alpha=8.0) -> TinyChatLM:
    require(not any(isinstance(m,LoRALinear) for m in base.modules()),
            '不允许重复注入LoRA')
    policy=copy.deepcopy(base)
    policy.requires_grad_(False)
    for block in policy.blocks:
        block.attn.q_proj=LoRALinear(block.attn.q_proj,rank,alpha)
        block.attn.v_proj=LoRALinear(block.attn.v_proj,rank,alpha)
    return policy


def adapters(model:TinyChatLM):
    return [(n,p) for n,p in model.named_parameters() if n.endswith(('.A','.B'))]


def adapter_snapshot(model:TinyChatLM):
    return {n:p.detach().cpu().clone() for n,p in adapters(model)}


def restore_adapter(model:TinyChatLM,values:dict):
    parameters=dict(model.named_parameters())
    require(set(values)==set(adapter_snapshot(model)),'adapter键集合不匹配')
    with torch.no_grad():
        for name,v in values.items():
            require(v.shape==parameters[name].shape and bool(torch.isfinite(v).all()),
                    'adapter参数形状/数值不匹配')
            parameters[name].copy_(v.to(parameters[name]))


def base_state(model:TinyChatLM):
    base={}
    for name,value in model.state_dict().items():
        if name.endswith(('.A','.B')):continue
        normalized=name.replace('.q_proj.base.','.q_proj.').replace('.v_proj.base.','.v_proj.')
        require(normalized not in base,'归一化基座命名冲突')
        base[normalized]=value
    return base


def base_hash(model:TinyChatLM):
    return state_hash(base_state(model))

In [11]:
seed_all(6901)
base_demo=TinyChatLM(ToyConfig(tok.vocab_size)).eval()
policy_demo=inject_lora(base_demo)
print('完整基座参数:',sum(p.numel() for p in base_demo.parameters()))
print('可训练LoRA参数:',sum(p.numel() for _,p in adapters(policy_demo)))
print('可训练参数名字:',[n for n,p in policy_demo.named_parameters() if p.requires_grad])
with torch.no_grad():
    logits_base=base_demo(batch['input_ids'],batch['attention_mask'])
    logits_policy=policy_demo(batch['input_ids'],batch['attention_mask'])
print('初始policy与基座最大logits差:',(logits_base-logits_policy).abs().max().item())

完整基座参数: 40000
可训练LoRA参数: 1024
可训练参数名字: ['blocks.0.attn.q_proj.A', 'blocks.0.attn.q_proj.B', 'blocks.0.attn.v_proj.A', 'blocks.0.attn.v_proj.B', 'blocks.1.attn.q_proj.A', 'blocks.1.attn.q_proj.B', 'blocks.1.attn.v_proj.A', 'blocks.1.attn.v_proj.B']
初始policy与基座最大logits差: 0.0


## 6. 梯度连通：首步 A 为零，不意味着训练有错

因为 B 初始化为0，首步 LoRA 对 A 的梯度通常是零，但 B 会收到梯度。B 发生更新以后，后续 A 才开始收到梯度。

必须分别检查：

- 首步 B 的梯度确实非零；
- 后续 A 开始有梯度；
- 冻结基座没有梯度，也没有参数变化；
- `optimizer.param_groups` **恰好**覆盖 A/B，不因为 `requires_grad=False` 就允许错误优化器组；
- 不把 `model.eval()` 当成“关闭自动微分”。

Notebook 后面的完整检查与训练历史会打印这些证据。

## 7. 监督 loss 的移位、分母与 EOS

本课模型只输出 logits，不在内部计算 loss，所以把 `logits[:, :-1]` 对齐 `labels[:, 1:]`；忽略 -100；**按所有有效回答token总数**归一化，不按题目数或 padding 后的矩形宽度。

例如有效目标有2个和4个，两条样本合起来的分母是6。EOS包含在有效目标里。真实 HF 的 `AutoModelForCausalLM(...,labels=...)` 通常自己做 shift；可选烟雾测试会将其返回 loss 与外部明确移位公式数值核对，而不是让两路重复移位。([HF 训练任务说明](https://huggingface.co/docs/transformers/main/tasks/language_modeling))

In [12]:
def supervised_token_loss(logits:torch.Tensor,labels:torch.Tensor) -> tuple[torch.Tensor,dict]:
    """模型仅返回logits，在此处shift一次，返回token平均SFT loss。"""
    require(logits.ndim==3 and labels.shape==logits.shape[:2] and labels.dtype==torch.long,
            'logits/labels形状type错误')
    require(logits.shape[1]>1 and bool((labels[:,0]==IGNORE).all()),'首位置必须忽略')
    require(bool(torch.isfinite(logits).all()),'非有限logits')
    targets=labels[:,1:];valid=targets!=IGNORE
    count=int(valid.sum())
    require(count>0,'全忽略不能训练')
    require(bool(((targets[valid]>=0)&(targets[valid]<logits.shape[-1])).all()),
            '监督目标越界')
    # 显式记录分子/分母，支持变长batch和累积训练时正确归一化。
    token_losses=F.cross_entropy(logits[:,:-1,:].reshape(-1,logits.shape[-1]),
                                 targets.reshape(-1),ignore_index=IGNORE,reduction='none')
    total=token_losses.sum();loss=total/count
    return loss,{'target_tokens':count,'loss_sum':total.detach().item(),
                 'mean_nll':loss.detach().item()}


@dataclass(frozen=True)
class ToyTrainConfig:
    seed: int=6901
    steps: int=16
    batch_size: int=8
    lr: float=0.02
    max_length: int=160
    rank: int=4
    alpha: float=8.0
    max_grad_norm: float=1.0
    static_budget_mib: float=64.0
    cuda_peak_budget_mib: float=0.0

    def validate(self):
        require(type(self.seed) is int and self.seed>=0,'seed非法')
        for key in ('steps','batch_size','max_length','rank'):
            require(type(getattr(self,key)) is int and getattr(self,key)>0,key+'必须正整数')
        for key in ('lr','alpha','max_grad_norm','static_budget_mib'):
            value=getattr(self,key)
            require(type(value) in (int,float) and math.isfinite(value) and value>0,
                    key+'必须有限正数')
        require(type(self.cuda_peak_budget_mib) in (int,float) and
                math.isfinite(self.cuda_peak_budget_mib) and self.cuda_peak_budget_mib>=0,
                'cuda_peak_budget_mib非法')


def memory_accounting(policy:TinyChatLM,batch_size:int,seq_length:int)->dict:
    """仅为静态分项核算，不是训练峰值的上下界承诺。"""
    require(type(batch_size)is int and batch_size>0 and type(seq_length)is int and seq_length>0,
            'batch/seq必须正整数')
    base_bytes=sum(p.numel()*p.element_size() for p in policy.parameters() if not p.requires_grad)
    adapter_bytes=sum(p.numel()*p.element_size() for _,p in adapters(policy))
    grad_bytes=adapter_bytes
    # FP32 Adam状态的两个一阶/二阶矩仅是示例，另有可能的临时工作区。
    adam_moments=2*adapter_bytes
    b,t=batch_size,seq_length
    activations_proxy=b*t*policy.cfg.dim*4
    attention_proxy=b*policy.cfg.heads*t*t*4*policy.cfg.layers
    output_proxy=b*t*policy.cfg.vocab_size*4
    static=base_bytes+adapter_bytes+grad_bytes+adam_moments
    return {'dtype':'fp32_toy','frozen_parameter_bytes':base_bytes,
            'trainable_parameter_bytes':adapter_bytes,'adapter_gradient_bytes':grad_bytes,
            'estimated_adam_moment_bytes':adam_moments,
            'bookkept_static_bytes':static,'bookkept_static_mib':static/2**20,
            'activation_example_bytes':activations_proxy,
            'attention_example_bytes':attention_proxy,
            'logits_example_bytes':output_proxy,
            'warning':'static accounting/illustrative activation sizes; NOT measured peak, upper bound or GPU reservation'}


def cuda_metrics(device:torch.device) -> dict:
    if device.type!='cuda':
        return {'device':'cpu','peak_allocated_mib':None,'peak_reserved_mib':None,
                'free_device_bytes':None,'measured_cuda':False}
    torch.cuda.synchronize(device)
    free,total=torch.cuda.mem_get_info(device)
    return {'device':str(device),'peak_allocated_mib':torch.cuda.max_memory_allocated(device)/2**20,
            'peak_reserved_mib':torch.cuda.max_memory_reserved(device)/2**20,
            'free_device_bytes':int(free),'total_device_bytes':int(total),
            'measured_cuda':True,'warning':'allocator peak not total process/GPU peak or reservation guarantee'}


@torch.no_grad()
def evaluate_nll(model:TinyChatLM,rows:list[dict],tok:ByteMergeChatTokenizer,
                 max_length=160,batch_size=8)->dict:
    require(bool(rows),'评估集合不能空')
    mode=model.training;model.eval()
    device=next(model.parameters()).device
    total_loss=0.0;total_tokens=0
    try:
        for start in range(0,len(rows),batch_size):
            chunk=[encode_training_row(tok,r,max_length) for r in rows[start:start+batch_size]]
            b=collate_training_rows(chunk,tok,device)
            value,parts=supervised_token_loss(model(b['input_ids'],b['attention_mask']),b['labels'])
            total_loss+=parts['loss_sum'];total_tokens+=parts['target_tokens']
    finally:model.train(mode)
    return {'examples':len(rows),'target_tokens':total_tokens,
            'teacher_forced_nll':total_loss/total_tokens,
            'not_generation_accuracy':True}

In [13]:
CONFIG = ToyTrainConfig()
print('默认训练配置:',asdict(CONFIG))

默认训练配置: {'seed': 6901, 'steps': 16, 'batch_size': 8, 'lr': 0.02, 'max_length': 160, 'rank': 4, 'alpha': 8.0, 'max_grad_norm': 1.0, 'static_budget_mib': 64.0, 'cuda_peak_budget_mib': 0.0}


In [14]:
uniform=torch.zeros(2,5,4,dtype=torch.float64,requires_grad=True)
labels=torch.tensor([[-100,-100,2,3,-100],[-100,1,2,3,0]])
loss,details=supervised_token_loss(uniform,labels)
print('有效目标token数:',details['target_tokens'])
print('loss:',float(loss),'手算:',math.log(4))
loss.backward()
print('忽略提示的logits梯度:',uniform.grad[0,0].tolist())

有效目标token数: 6
loss: 1.3862943611198906 手算: 1.3862943611198906
忽略提示的logits梯度: [0.0, 0.0, 0.0, 0.0]


/tmp/ipykernel_964/1474595838.py:5: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print('loss:',float(loss),'手算:',math.log(4))


## 8. 显存预算：静态核算≠实际训练峰值

本课先列出基座参数、LoRA参数、LoRA梯度与示例 FP32 Adam 一二阶矩字节，再展示随 B、T 增长的 activation / attention / logits **单项示意体积**。

注意：

- **不是**显存总量、峰值的严谨上界或GPU能否跑通的保证；还有算子工作区、kernel、框架缓存、CUDA上下文、内存碎片及额外副本。
- CPU 路径返回 CUDA 峰值为 `None`，**不会用“0 MiB”冒充实测**。
- 在真实 CUDA 上执行时，使用 `max_memory_allocated()` 和 `max_memory_reserved()` 记录不同口径；它们也不等于整卡被占用的全部显存。[PyTorch allocated](https://docs.pytorch.org/docs/stable/generated/torch.cuda.memory.max_memory_allocated.html) · [reserved](https://docs.pytorch.org/docs/stable/generated/torch.cuda.memory.max_memory_reserved.html)
- 静态闸门在训练前使用；峰值检查在运行后才能判断，**不是防止运行中 OOM 的硬保证**。

In [15]:
accounting=memory_accounting(policy_demo,batch_size=8,seq_length=100)
for name in ['frozen_parameter_bytes','trainable_parameter_bytes','adapter_gradient_bytes',
             'estimated_adam_moment_bytes','bookkept_static_mib','attention_example_bytes','logits_example_bytes']:
    print(name,':',accounting[name])
print('CPU设备报告:',cuda_metrics(torch.device('cpu')))
print('上下文T翻倍，attention示意开销倍率:',
      memory_accounting(policy_demo,8,200)['attention_example_bytes']/accounting['attention_example_bytes'])

frozen_parameter_bytes : 160000
trainable_parameter_bytes : 4096
adapter_gradient_bytes : 4096
estimated_adam_moment_bytes : 8192
bookkept_static_mib : 0.168212890625
attention_example_bytes : 2560000
logits_example_bytes : 899200
CPU设备报告: {'device': 'cpu', 'peak_allocated_mib': None, 'peak_reserved_mib': None, 'free_device_bytes': None, 'measured_cuda': False}
上下文T翻倍，attention示意开销倍率: 4.0


## 9. Adapter保存与基座身份：不能只靠相同架构

本课分别保存 toy 基座和 adapter，adapter metadata 带有**基座参数内容哈希、架构配置、分词约定、rank与alpha**。

加载时明确要求基座一致；架构同为隐藏维度32，不代表参数值相同。还要检查 adapter 的名称、形状、有限性。只加载本课生成的可信本地 `.pt`，调用 `weights_only=True`。真实 PEFT 的 `save_pretrained()` 通常只保存适配器，因此重新使用时必须对准合适的基座。[PEFT Quicktour](https://huggingface.co/docs/peft/quicktour)

教学 toy 保存完整基座只是为了证明独立重载；它不会伪称这些 `.pt` 已经是通用 HF/PEFT checkpoint，也不保存优化器/RNG等精确续训状态。

In [16]:
def toy_adapter_step(policy:TinyChatLM,optimizer,rows:list[dict],
                     tok:ByteMergeChatTokenizer,max_length=160,max_grad_norm=1.0) -> dict:
    require(all(r['split']=='train' for r in rows),'更新入口只接受train')
    trainable={id(p) for _,p in adapters(policy)}
    optimizer_params=[p for group in optimizer.param_groups for p in group['params']]
    require(len(optimizer_params)==len(trainable) and
            {id(p) for p in optimizer_params}==trainable,
            '优化器必须恰好含LoRA A/B，不可含冻结基座')
    require(all(p.requires_grad == n.endswith(('.A','.B'))
                for n,p in policy.named_parameters()),'只有LoRA A/B允许梯度')
    require(type(max_grad_norm) in (int,float) and math.isfinite(max_grad_norm)
            and max_grad_norm>0,'梯度阈值非法')
    device=next(policy.parameters()).device
    encoded=[encode_training_row(tok,r,max_length) for r in rows]
    batch=collate_training_rows(encoded,tok,device)
    policy.train()
    optimizer.zero_grad(set_to_none=True)
    loss,parts=supervised_token_loss(policy(batch['input_ids'],batch['attention_mask']),batch['labels'])
    loss.backward()
    gradients={n:float(p.grad.norm()) for n,p in adapters(policy) if p.grad is not None}
    require(len(gradients)==len(trainable) and all(math.isfinite(x) for x in gradients.values()),
            'LoRA梯度未连通或包含非有限值')
    require(all(p.grad is None for n,p in policy.named_parameters() if not n.endswith(('.A','.B'))),
            '冻结基座收到梯度')
    norm=nn.utils.clip_grad_norm_([p for _,p in adapters(policy)],
                                   max_grad_norm,error_if_nonfinite=True)
    optimizer.step()
    return {'loss':float(loss.detach()),'tokens':parts['target_tokens'],
            'grad_norm_pre_clip':float(norm),'A_grad_sum':sum(v for k,v in gradients.items() if k.endswith('.A')),
            'B_grad_sum':sum(v for k,v in gradients.items() if k.endswith('.B'))}


def save_toy_bundle(folder:Path,base:TinyChatLM,policy:TinyChatLM,
                    tok:ByteMergeChatTokenizer,data:dict,cfg:ToyTrainConfig,report:dict):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    require(not any(folder.iterdir()),'输出目录必须为空；不覆盖历史adapter')
    base_data={k:v.detach().cpu().clone() for k,v in base.state_dict().items()}
    base_id=state_hash(base_data)
    require(base_hash(policy)==base_id,'adapter必须依附该精确基座')
    torch.save({'version':VERSION,'kind':'toy_base','model_config':asdict(base.cfg),
                'tokenizer':tok.contract(),'base_hash':base_id,'state':base_data},folder/'toy_base.pt')
    torch.save({'version':VERSION,'kind':'toy_lora','model_config':asdict(base.cfg),
                'tokenizer':tok.contract(),'base_hash':base_id,'rank':cfg.rank,
                'alpha':cfg.alpha,'state':adapter_snapshot(policy)},folder/'toy_adapter.pt')
    write_json(folder/'data.json',data)
    write_json(folder/'report.json',report)


def load_toy_bundle(folder:Path,device='cpu'):
    folder=Path(folder)
    base_blob=torch.load(folder/'toy_base.pt',map_location='cpu',weights_only=True)
    adapter_blob=torch.load(folder/'toy_adapter.pt',map_location='cpu',weights_only=True)
    tok=ByteMergeChatTokenizer()
    require(base_blob['version']==adapter_blob['version']==VERSION,'检查点版本不匹配')
    require(base_blob['kind']=='toy_base' and adapter_blob['kind']=='toy_lora','文件类型错配')
    require(base_blob['model_config']==adapter_blob['model_config'],'架构配置不同')
    require(base_blob['tokenizer']==adapter_blob['tokenizer']==tok.contract(),
            '分词器/模板契约不同')
    require(state_hash(base_blob['state'])==base_blob['base_hash']==adapter_blob['base_hash'],
            'LoRA依赖的基座身份不匹配')
    base=TinyChatLM(ToyConfig(**base_blob['model_config'])).to(device)
    base.load_state_dict(base_blob['state'],strict=True)
    base.requires_grad_(False).eval()
    policy=inject_lora(base,rank=adapter_blob['rank'],alpha=adapter_blob['alpha'])
    restore_adapter(policy,adapter_blob['state'])
    policy.requires_grad_(False).eval()
    return base,policy,tok


def new_run_dir(root:Path) -> Path:
    root=Path(root)
    root.mkdir(parents=True,exist_ok=True)
    folder=root/('run_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    folder.mkdir(parents=False,exist_ok=False)
    return folder

## 10. 默认离线训练：真实前向、反向，冻结权重逐位保持

执行完整流程：数据校验 → 模型/adapter初始化 → 首次 logit 恒等 → 静态核算资源闸门 → 16次训练 → 检查冻结基座和梯度 → 对 train/dev/test 进行 teacher-forced NLL 报告 → 保存基座和adapter → 独立对象重载。

**只报告给定参考回答的条件 NLL**，不将其称作准确率、自然语言泛化、JSON 自由生成成功率。训练目标默认从随机初始化开始，不能把它与真实预训练Qwen的能力混为一谈。

训练轮次和最终模型选择预先固定为最后一步，不拿测试集来选超参。

In [17]:
def run_offline_preflight(output_root='lesson_69_outputs',device_name='cpu',
                          cfg:ToyTrainConfig|None=None) -> tuple[Path,dict]:
    cfg=cfg or ToyTrainConfig();cfg.validate()
    device=device_from_name(device_name)
    torch.set_num_threads(1)
    seed_all(cfg.seed)
    tok=ByteMergeChatTokenizer();data=make_dataset(cfg.seed)
    encoded={s:[encode_training_row(tok,r,cfg.max_length) for r in rows]
             for s,rows in data.items()}
    input_lengths=[len(e['input_ids']) for rows in encoded.values() for e in rows]
    label_counts=[e['answer_len'] for rows in encoded.values() for e in rows]
    require(max(input_lengths)<=cfg.max_length and min(label_counts)>=2,
            '训练输入未通过序列边界审计')
    model_cfg=ToyConfig(tok.vocab_size,max_length=cfg.max_length)
    base=TinyChatLM(model_cfg).to(device).requires_grad_(False).eval()
    frozen_id=state_hash(base.state_dict())
    seed_all(cfg.seed+1)
    policy=inject_lora(base,cfg.rank,cfg.alpha).to(device)
    adapter_initial=state_hash(adapter_snapshot(policy))
    b0=collate_training_rows(encoded['train'][:cfg.batch_size],tok,device)
    with torch.no_grad():
        before_logits=base(b0['input_ids'],b0['attention_mask'])
        after_inject=policy(b0['input_ids'],b0['attention_mask'])
    inject_diff=(before_logits-after_inject).abs().max().item()
    require(inject_diff < 1e-6,'初始LoRA并非零增量')
    require(base_hash(policy)==frozen_id,'注入LoRA意外改变基座')
    estimate=memory_accounting(policy,cfg.batch_size,max(input_lengths))
    require(estimate['bookkept_static_mib']<=cfg.static_budget_mib,
            '静态核算超过用户资源预算，未执行训练')
    if device.type=='cuda':torch.cuda.reset_peak_memory_stats(device)
    dev_before=evaluate_nll(policy,data['dev'],tok,cfg.max_length)
    train_before=evaluate_nll(policy,data['train'],tok,cfg.max_length)
    opt=torch.optim.AdamW([p for _,p in adapters(policy)],lr=cfg.lr,weight_decay=0.0)
    rng=random.Random(cfg.seed+2)
    history=[]
    for step in range(cfg.steps):
        sample=rng.sample(data['train'],min(cfg.batch_size,len(data['train'])))
        info=toy_adapter_step(policy,opt,sample,tok,cfg.max_length,cfg.max_grad_norm)
        info['step']=step+1
        history.append(info)
    require(base_hash(policy)==frozen_id and state_hash(base.state_dict())==frozen_id,
            '训练不允许修改基座')
    require(state_hash(adapter_snapshot(policy))!=adapter_initial,
            'LoRA在训练后仍完全不变')
    require(all(p.grad is None for p in base.parameters()),'基座积累了梯度')
    dev_after=evaluate_nll(policy,data['dev'],tok,cfg.max_length)
    train_after=evaluate_nll(policy,data['train'],tok,cfg.max_length)
    # 无模型能力宣称：这里只比较给定参考回答的teacher-forced NLL。
    test_reference=evaluate_nll(base,data['test'],tok,cfg.max_length)
    test_after=evaluate_nll(policy,data['test'],tok,cfg.max_length)
    gpu=cuda_metrics(device)
    if device.type=='cuda' and cfg.cuda_peak_budget_mib>0:
        require(gpu['peak_allocated_mib']<=cfg.cuda_peak_budget_mib,
                '实测CUDA已分配峰值超出设定值（非预防性上限）')
    report={'version':VERSION,'environment':{'python':platform.python_version(),
                'torch':str(torch.__version__),'device':str(device),'seed':cfg.seed},
            'config':asdict(cfg),'model':asdict(model_cfg),
            'dataset_counts':{s:len(rows) for s,rows in data.items()},
            'dataset_digest':digest(data),'tokenizer_digest':digest(tok.contract()),
            'train_seq_min_max':[min(input_lengths),max(input_lengths)],
            'answer_token_min_max':[min(label_counts),max(label_counts)],
            'initial_logits_max_diff':inject_diff,
            'base_hash_unchanged':True,'base_hash':frozen_id,
            'all_trainable_are_lora':True,'reference_grad_none':True,
            'base_parameters':sum(p.numel() for p in base.parameters()),
            'adapter_parameters':sum(p.numel() for _,p in adapters(policy)),
            'static_memory_accounting':estimate,'cuda_memory':gpu,
            'train_before':train_before,'train_after':train_after,
            'dev_before':dev_before,'dev_after':dev_after,
            'test_frozen_base':test_reference,'test_adapter':test_after,
            'updates':history,'selection_rule':'fixed final update; no test selection',
            'limitations':['toy_random_base_not_pretrained','synthetic_copy_task_not_generalization',
                'training_NLL_not_free_generation','one_seed','manual_LoRA_not_PEFT_export',
                'no_CUDA_measurement_on_cpu','static_memory_not_peak_upper_bound']}
    folder=new_run_dir(Path(output_root))
    save_toy_bundle(folder,base,policy,tok,data,cfg,report)
    reloaded_base,reloaded_policy,reloaded_tok=load_toy_bundle(folder,device)
    with torch.no_grad():
        old=policy(b0['input_ids'],b0['attention_mask'])
        restored=reloaded_policy(b0['input_ids'],b0['attention_mask'])
    reload_diff=(old-restored).abs().max().item()
    require(torch.allclose(old,restored,atol=1e-6,rtol=1e-5),'独立对象重载输出不一致')
    require(state_hash(reloaded_base.state_dict())==frozen_id,'独立重载基座不同')
    require(all(not p.requires_grad for p in reloaded_policy.parameters()),
            '推理模型仍有可训练参数')
    reload_report={'max_logit_abs_diff':reload_diff,
                   'base_hash_equal':True,
                   'adapter_hash_equal':state_hash(adapter_snapshot(policy))==state_hash(adapter_snapshot(reloaded_policy)),
                   'trainable_after_reload':sum(p.numel() for p in reloaded_policy.parameters() if p.requires_grad),
                   'tokenizer_equal':reloaded_tok.contract()==tok.contract(),
                   'scope':'toy_trusted_local_weights_only_not_exact_training_resume'}
    require(reload_report['adapter_hash_equal'],'adapter加载参数不同')
    write_json(folder/'reload.json',reload_report)
    print('toy基座参数:',report['base_parameters'],'LoRA参数:',report['adapter_parameters'])
    print('训练侧NLL:',round(train_before['teacher_forced_nll'],6),'→',round(train_after['teacher_forced_nll'],6))
    print('开发侧NLL:',round(dev_before['teacher_forced_nll'],6),'→',round(dev_after['teacher_forced_nll'],6))
    print('输出目录:',folder)
    return folder,report

In [18]:
RUN_DIR = None
report = None
if RUN_FULL_TOY:
    RUN_DIR,report=run_offline_preflight(OUTPUT_ROOT,device_name=DEVICE,cfg=CONFIG)
else:
    print('已经明确跳过训练；本次不生成假训练结果。')

toy基座参数: 40000 LoRA参数: 1024
训练侧NLL: 5.611598 → 4.39499
开发侧NLL: 5.611675 → 4.390481
输出目录: lesson_69_outputs/run_20261104T034126_850237Z


In [19]:
if report is not None:
    for label,key in [('训练前','train_before'),('训练后','train_after'),
                      ('开发前','dev_before'),('开发后','dev_after'),
                      ('测试基座','test_frozen_base'),('测试adapter','test_adapter')]:
        print(label,':',round(report[key]['teacher_forced_nll'],6),
              '监督token数:',report[key]['target_tokens'])
    print('首步 A/B 梯度范数和:',report['updates'][0]['A_grad_sum'],report['updates'][0]['B_grad_sum'])
    print('第二步 A/B 梯度范数和:',report['updates'][1]['A_grad_sum'],report['updates'][1]['B_grad_sum'])
    print('基座冻结保持:',report['base_hash_unchanged'], 'CUDA已实测?',report['cuda_memory']['measured_cuda'])

训练前 : 5.611598 监督token数: 224
训练后 : 4.39499 监督token数: 224
开发前 : 5.611675 监督token数: 56
开发后 : 4.390481 监督token数: 56
测试基座 : 5.611589 监督token数: 56
测试adapter : 4.397205 监督token数: 56
首步 A/B 梯度范数和: 0.0 0.08266906160861254
第二步 A/B 梯度范数和: 0.04883875045925379 0.08205713238567114
基座冻结保持: True CUDA已实测? False


## 11. 可选：本地真实 HF+PEFT 单步梯度检查（**默认不执行**）

只有你已准备好**可信、完整、兼容、在可承受资源内**的本地 Safetensors checkpoint，并预先安装兼容的 `transformers`、`peft`、`safetensors` 才能运行。

额外约束：

1. 不接受仓库名代替本地目录；不用联网补齐缺失文件。
2. `local_files_only=True`、`trust_remote_code=False`、`use_safetensors=True`；不执行远端自定义代码。
3. 必须有真实 tokenizer Chat Template；检查 assistant 生成头、完整训练 token 序列的前缀关系与 EOS。
4. `target_modules` 必须明确匹配实际存在的 `nn.Linear`；没有匹配就停止，不猜 Qwen/Llama 架构。
5. 冻结基座、只训练 PEFT LoRA，执行**一步**前向/反向/更新，然后再次哈希冻结参数与本地源权重文件。
6. 输出必须写在模型目录之外。默认的本地权重大小闸门为256 MiB，不保证这个上限就不会OOM。

**制作本 Notebook 的环境没有 transformers/peft，因此该可选分支没有实际加载真实 HF 模型测试。** 这段是依赖条件明确的可选入口，不是已完成的真实模型训练结果。`local_files_only` 等选项不是安全沙箱；请只对可信本地文件操作。[HF 离线](https://huggingface.co/docs/transformers/installation) · [PEFT 注入](https://huggingface.co/docs/peft/quicktour)

In [20]:
def inspect_local_training_candidate(path:str|Path,max_weight_bytes:int=2*1024**3,
                                     full_weight_hash=False) -> dict:
    result=inspect_local_checkpoint(path,full_weight_hash=full_weight_hash,
                                    max_weight_bytes=max_weight_bytes)
    folder=ensure_local_directory(path)
    config=read_limited_json(folder/'config.json')
    require(config.get('is_encoder_decoder',False) is False,'本课仅接受decoder-only')
    require(any((folder/name).is_file() and not (folder/name).is_symlink()
                for name in ('tokenizer.json','tokenizer.model','vocab.json','vocab.txt')),
            '没有本地tokenizer证据；不下载补齐')
    result['preflight']='metadata_only_does_not_load_or_train'
    result['needs_explicit_training_flag']=True
    result['warning']+='; local_files_only and trust_remote_code=False are not a sandbox'
    return result


def _hf_encode_example(tokenizer,row,max_length=160):
    """真实模型必须使用自身Chat Template，并逐token核对提示边界。"""
    require(getattr(tokenizer,'chat_template',None) is not None,
            '本地模型tokenizer缺少chat_template，不自动猜拼接规则')
    require(getattr(tokenizer,'eos_token_id',None) is not None,'EOS未定义')
    messages=messages_for_row(row)
    prefix=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True)
    full=tokenizer.apply_chat_template(messages+[{'role':'assistant','content':row['assistant']}],
                                       tokenize=True,add_generation_prompt=False)
    require(type(prefix) is list and type(full) is list and
            all(type(x) is int for x in full+prefix),'需要一维token ID列表')
    require(full[:len(prefix)]==prefix and len(full)>len(prefix),
            '真实模板的assistant header与完整训练序列token边界不一致，不能猜损失掩码')
    eos=tokenizer.eos_token_id
    require(type(eos) is int and full[-1]==eos,
            '真实模板不以tokenizer.eos_token_id结束；须先确定模型停止契约')
    require(len(full)<=max_length,'真实训练样本超过长度；本课不执行截断')
    require(len(full)-len(prefix)>1,'没有完整的回答加EOS')
    return {'input_ids':full,'labels':[IGNORE]*len(prefix)+full[len(prefix):],
            'prompt_len':len(prefix),'answer_len':len(full)-len(prefix)}


def run_local_peft_smoke(path:str|Path,output_root:str|Path,
                         *,device_name='cpu',max_weight_bytes=256*2**20,
                         target_modules=('q_proj','v_proj'),rank=4,alpha=8.0,
                         max_length=160) -> tuple[Path,dict]:
    """显式本地训练；无可信小模型及依赖时不能执行。默认完全不调用。"""
    import importlib.util
    require(importlib.util.find_spec('transformers') is not None and
            importlib.util.find_spec('peft') is not None and
            importlib.util.find_spec('safetensors') is not None,
            '本地HF烟雾测试需要额外安装兼容的transformers/peft/safetensors；不会自动安装')
    require(type(rank) is int and rank>0 and type(alpha) in (float,int) and alpha>0,
            'LoRA配置不合法')
    require(type(max_length) is int and max_length>=8,'max_length非法')
    require(type(target_modules) in (list,tuple) and bool(target_modules) and
            len(set(target_modules))==len(target_modules) and
            all(type(n) is str and n and n.replace('_','').isalnum() for n in target_modules),
            'target_modules必须是明确无重复的层后缀')
    folder=ensure_local_directory(path)
    validate_output_outside_model(folder,output_root)
    gate=inspect_local_training_candidate(folder,max_weight_bytes=max_weight_bytes,
                                           full_weight_hash=True)
    device=device_from_name(device_name)
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig,TaskType,get_peft_model
    # 用户明确同意可信本地仓库；不开远端请求，不允许远端自定义Python类。
    tokenizer=AutoTokenizer.from_pretrained(str(folder),local_files_only=True,
                                            trust_remote_code=False,use_fast=True)
    data=make_dataset()
    encoded=[_hf_encode_example(tokenizer,r,max_length) for r in data['train'][:2]]
    if tokenizer.pad_token_id is None:
        require(type(tokenizer.eos_token_id) is int,'pad与eos均不可用')
        pad_id=tokenizer.eos_token_id
    else:pad_id=tokenizer.pad_token_id
    width=max(len(s['input_ids']) for s in encoded)
    ids=torch.tensor([s['input_ids']+[pad_id]*(width-len(s['input_ids'])) for s in encoded],
                     dtype=torch.long,device=device)
    mask=torch.tensor([[1]*len(s['input_ids'])+[0]*(width-len(s['input_ids'])) for s in encoded],
                      dtype=torch.long,device=device)
    labels=torch.tensor([s['labels']+[IGNORE]*(width-len(s['labels'])) for s in encoded],
                        dtype=torch.long,device=device)
    # dtype参数是当前HF版本的标准写法；依赖版本不兼容应明确抛异常。
    base=AutoModelForCausalLM.from_pretrained(str(folder),local_files_only=True,
             trust_remote_code=False,use_safetensors=True,dtype=torch.float32)
    base=base.to(device)
    targets=[(name,m) for name,m in base.named_modules()
             if any(name==suffix or name.endswith('.'+suffix) for suffix in target_modules)]
    require(bool(targets) and all(isinstance(m,nn.Linear) for _,m in targets),
            '本地模型找不到全部适配的nn.Linear目标；需审计架构并显式调整targets')
    for suffix in target_modules:
        require(any(name==suffix or name.endswith('.'+suffix) for name,_ in targets),
                '目标模块没有匹配：'+suffix)
    config=LoraConfig(r=rank,lora_alpha=alpha,lora_dropout=0.0,bias='none',
                      target_modules=list(target_modules),task_type=TaskType.CAUSAL_LM)
    policy=get_peft_model(base,config)
    trainable=[(n,p) for n,p in policy.named_parameters() if p.requires_grad]
    require(bool(trainable) and all(('lora_A.' in n or 'lora_B.' in n) for n,_ in trainable),
            '预检发现非LoRA参数可训练')
    frozen=[(n,p) for n,p in policy.named_parameters() if not p.requires_grad]
    before_hash=state_hash(dict(frozen))
    policy.train()
    optimizer=torch.optim.AdamW([p for _,p in trainable],lr=1e-4,weight_decay=0.0)
    if device.type=='cuda':torch.cuda.reset_peak_memory_stats(device)
    optimizer.zero_grad(set_to_none=True)
    result=policy(input_ids=ids,attention_mask=mask,labels=labels)
    require(result.loss is not None and bool(torch.isfinite(result.loss)),
            'HF模型不支持可用的CausalLM损失')
    # HF的loss通常已执行内部shift；这里数值比对，不再次对loss移位。
    manual=F.cross_entropy(result.logits[:,:-1,:].contiguous().view(-1,result.logits.shape[-1]).float(),
                           labels[:,1:].contiguous().view(-1),ignore_index=IGNORE)
    require(torch.allclose(result.loss.float(),manual,rtol=2e-3,atol=2e-3),
            '模型内部shift/归约与当前标签契约不一致，停止')
    result.loss.backward()
    require(all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for _,p in trainable),
            'PEFT adapter未形成完整梯度')
    require(all(p.grad is None for _,p in frozen),'冻结HF基座收到了梯度')
    norm=nn.utils.clip_grad_norm_([p for _,p in trainable],1.0,error_if_nonfinite=True)
    optimizer.step()
    require(state_hash(dict(frozen))==before_hash,'真实本地HF基座意外更新')
    end_hashes=inspect_local_training_candidate(folder,max_weight_bytes=max_weight_bytes,
                                                full_weight_hash=True)
    require(gate['weight_files']==end_hashes['weight_files'] and
            gate['small_files']==end_hashes['small_files'],
            '本地源checkpoint文件被修改')
    output=new_run_dir(Path(output_root))
    adapter_dir=output/'adapter'
    policy.save_pretrained(adapter_dir,safe_serialization=True)
    measured=cuda_metrics(device)
    report={'version':VERSION,'mode':'explicit_local_peft_smoke',
            'confirmed_pretrained_quality':False,'source_audit':gate,
            'source_weight_files_unchanged':True,
            'target_modules':[n for n,_ in targets],
            'trainable_parameters':sum(p.numel() for _,p in trainable),
            'trainable_names':[n for n,_ in trainable],
            'loss_before_one_update':float(result.loss.detach()),
            'HF_loss_matches_manual_shift':True,
            'gradient_norm_pre_clip':float(norm),
            'frozen_base_unchanged':True,'tokenizer_prefix_matched':True,
            'memory':measured,'limits':[
                'not_a_model_quality_test','single_gradient_step_only',
                'not_tested_on_local_HF_in_artifact_build_environment',
                'no_quantization_no_distributed_no_automatic_download',
                'not_a_security_sandbox']}
    write_json(output/'local_smoke_report.json',report)
    return output,report

In [21]:
RUN_LOCAL_PEFT_SMOKE = False
TRUSTED_LOCAL_MODEL_DIR = None  # 自行填写绝对本地目录；默认不要运行。
if RUN_LOCAL_PEFT_SMOKE:
    require(TRUSTED_LOCAL_MODEL_DIR is not None,'请先显式设置受信任本地模型目录')
    out, local_report=run_local_peft_smoke(TRUSTED_LOCAL_MODEL_DIR,OUTPUT_ROOT,device_name=DEVICE)
    print('本地PEFT单步检查:',out,local_report['frozen_base_unchanged'])
else:
    print('真实HF+PEFT分支未运行；不声称实际模型通过。')

真实HF+PEFT分支未运行；不声称实际模型通过。


## 12. 独立重载与反例

下一格从磁盘载入全新基座对象、注入相同 LoRA 结构并装入 adapter，再在同一批输入上核对 logits 与测试 NLL。它不会复用训练时活动模型的引用。

另外，固定测试会故意将adapter配到**结构相同但内容不同的基座**，应该拒绝而不是继续微调。我们也会检查损坏 adapter、缺少 EOS、错误源划分、损失错位和优化器覆盖范围等故障。

In [22]:
def reload_toy_run(folder:Path,device_name='cpu') -> dict:
    device=device_from_name(device_name)
    base,policy,tok=load_toy_bundle(folder,device)
    data=json.loads((Path(folder)/'data.json').read_text(encoding='utf-8'))
    validate_dataset(data)
    report=json.loads((Path(folder)/'report.json').read_text(encoding='utf-8'))
    require(digest(data)==report['dataset_digest'],'数据内容不是对应实验')
    val=evaluate_nll(policy,data['test'],tok,report['config']['max_length'])
    diff=abs(val['teacher_forced_nll']-report['test_adapter']['teacher_forced_nll'])
    require(diff<1e-5,'重载后的测试NLL超出数值容差')
    return {'test_nll_abs_diff':diff,'base_hash_equal':state_hash(base.state_dict())==report['base_hash'],
            'adapter_trainable_parameters_after_load':sum(p.numel() for p in policy.parameters() if p.requires_grad),
            'scope':'trusted_toy_checkpoint_inference_only_not_exact_resume'}

In [23]:
if RUN_DIR is not None:
    print('独立磁盘对象检查:',json.dumps(reload_toy_run(RUN_DIR,DEVICE),ensure_ascii=False,indent=2))
    print('重载元数据:',(RUN_DIR/'reload.json').read_text(encoding='utf-8'))
else:
    print('本次没有磁盘检查点。')

独立磁盘对象检查: {
  "test_nll_abs_diff": 0.0,
  "base_hash_equal": true,
  "adapter_trainable_parameters_after_load": 0,
  "scope": "trusted_toy_checkpoint_inference_only_not_exact_resume"
}
重载元数据: {
  "adapter_hash_equal": true,
  "base_hash_equal": true,
  "max_logit_abs_diff": 0.0,
  "scope": "toy_trusted_local_weights_only_not_exact_training_resume",
  "tokenizer_equal": true,
  "trainable_after_reload": 0
}



## 13. 固定检查

测试要求不仅“代码能运行”，还要证明：来源互斥、prompt/assistant头、EOS、右padding、因果性、移位梯度、LoRA首步梯度、冻结参数身份、优化器精确参数集合、adapter独立恢复、Safetensors预检与资源预算边界。

长代码单元默认折叠，便于专注理解。但你可以展开并逐项调试。固定检查并不证明真实预训练模型能力，也不证明任何不可信checkpoint安全。

In [24]:
class Checks:
    def __init__(self):self.names=[]
    def ok(self,name,condition):
        if not bool(condition):raise AssertionError(name)
        self.names.append(name)
    def eq(self,name,a,b):self.ok(name,a==b)
    def close(self,name,a,b,atol=1e-8,rtol=1e-8):
        self.ok(name,torch.allclose(torch.as_tensor(a,dtype=torch.float64),
            torch.as_tensor(b,dtype=torch.float64),atol=atol,rtol=rtol))
    def rejects(self,name,fn):
        try:fn()
        except (LessonError,ValueError,TypeError,OSError,UnicodeError):
            self.names.append(name);return
        raise AssertionError(name+': 错误输入未被拒绝')


def fake_safetensors_bytes(value:float=0.25) -> bytes:
    header={'toy.weight':{'dtype':'F32','shape':[1],'data_offsets':[0,4]}}
    raw=canonical(header).encode('utf-8')
    raw+=b' '*((8-len(raw)%8)%8)
    return struct.pack('<Q',len(raw))+raw+struct.pack('<f',float(value))


def fake_hf_folder(folder:Path):
    folder.mkdir(parents=True,exist_ok=True)
    write_json(folder/'config.json',{'model_type':'toy_mock','is_encoder_decoder':False})
    (folder/'model.safetensors').write_bytes(fake_safetensors_bytes())
    write_json(folder/'tokenizer.json',{'fixture':'not_a_real_tokenizer'})


def run_checks() -> dict:
    torch.set_num_threads(1)
    seed_all(6910)
    c=Checks();tok=ByteMergeChatTokenizer();data=make_dataset()
    c.eq('固定划分32_8_8',[len(data[s]) for s in ('train','dev','test')],[32,8,8])
    c.eq('来源分组8_2_2',[len({x['source_id'] for x in data[s]}) for s in ('train','dev','test')],[8,2,2])
    c.ok('来源互斥',not ({x['source_id'] for x in data['train']} & {x['source_id'] for x in data['dev']}))
    c.ok('全部48提示唯一',len({r['user'] for x in data.values() for r in x})==48)
    c.ok('全部回答符合固定JSON',all(r['assistant'].startswith('{"answer":') for x in data.values() for r in x))
    modified=copy.deepcopy(data);modified['test'][0]['source_id']=data['train'][0]['source_id']
    c.rejects('跨分组来源拒绝',lambda:validate_dataset(modified))
    modified=copy.deepcopy(data);modified['test'][0]['user']=data['train'][0]['user']
    c.rejects('训练测试提示重复拒绝',lambda:validate_dataset(modified))
    modified=copy.deepcopy(data);modified['dev'][0]['assistant']=canonical({'answer':'99'})
    c.rejects('错误参考答案拒绝',lambda:validate_dataset(modified))
    modified=copy.deepcopy(data);modified['dev'][0]['split']='train'
    c.rejects('split标签错误拒绝',lambda:validate_dataset(modified))
    c.rejects('缺少划分拒绝',lambda:validate_dataset({'train':data['train']}))
    c.ok('用户字面控制符属于普通字节',all(x>=tok.byte_offset for x in tok.encode('<im_start>')))
    user_escape=tok.apply_chat_template([{'role':'user','content':'<im_start> invisible'}],tokenize=True,
                                         add_generation_prompt=True)
    c.eq('用户不能注入额外结构开始符',user_escape.count(tok.im_start_token_id),2)
    c.ok('BOS固定',user_escape[0]==tok.bos_token_id)
    c.rejects('不接受未知角色',lambda:tok.apply_chat_template([{'role':'tool','content':'abc'}]))
    c.rejects('不接受assistant头重复',lambda:tok.apply_chat_template([{'role':'assistant','content':'a'}],add_generation_prompt=True))
    sample=encode_training_row(tok,data['train'][0])
    c.ok('token前缀长度正',sample['prompt_len']>0)
    c.ok('answer token长度正',sample['answer_len']>1)
    c.eq('EOS监督一次',sample['labels'][-1],tok.eos_token_id)
    c.ok('提示全忽略',all(y==IGNORE for y in sample['labels'][:sample['prompt_len']]))
    c.ok('回答label与input同位',sample['labels'][sample['prompt_len']:]==sample['input_ids'][sample['prompt_len']:])
    c.ok('实际助手头在prompt末尾',sample['input_ids'][sample['prompt_len']-1]==tok.role_ids['assistant'])
    c.rejects('不裁剪超长EOS',lambda:encode_training_row(tok,data['train'][0],max_length=8))
    batch=collate_training_rows([sample,encode_training_row(tok,data['train'][1])],tok)
    c.ok('批次输入标签同形状',batch['input_ids'].shape==batch['labels'].shape)
    c.ok('批次右padding为PAD',bool((batch['input_ids'][~batch['attention_mask']]==tok.pad_token_id).all()))
    c.ok('padding标签-100',bool((batch['labels'][~batch['attention_mask']]==IGNORE).all()))
    c.ok('attention_mask是bool',batch['attention_mask'].dtype==torch.bool)
    invalid=copy.deepcopy(sample);invalid['labels'][-1]=IGNORE
    c.rejects('EOS丢失拒绝',lambda:collate_training_rows([invalid],tok))
    invalid=copy.deepcopy(sample);invalid['labels']=invalid['labels'][:-1]
    c.rejects('标签长度错位拒绝',lambda:collate_training_rows([invalid],tok))
    # 模型、因果性与完整的shift/梯度测试
    cfg=ToyConfig(tok.vocab_size,dim=16,heads=2,layers=2,ff_dim=32)
    base=TinyChatLM(cfg).eval()
    ref_hash=state_hash(base.state_dict())
    policy=inject_lora(base,rank=2,alpha=4)
    c.eq('总LoRA线性目标数',len([m for m in policy.modules() if isinstance(m,LoRALinear)]),4)
    c.eq('可训练对象恰好8个AB矩阵',len(adapters(policy)),8)
    c.ok('只有AB可训练',all(p.requires_grad==n.endswith(('.A','.B')) for n,p in policy.named_parameters()))
    c.eq('基座状态哈希不变',base_hash(policy),ref_hash)
    c.ok('实际基座与策略独立存储',base.embed.weight.data_ptr()!=policy.embed.weight.data_ptr())
    with torch.no_grad():
        before=base(batch['input_ids'],batch['attention_mask'])
        initial=policy(batch['input_ids'],batch['attention_mask'])
    c.close('初始LoRA严格恒等',initial,before,atol=0,rtol=0)
    c.rejects('不允许二次LoRA注入',lambda:inject_lora(policy))
    c.rejects('非法rank拒绝',lambda:LoRALinear(nn.Linear(3,3),rank=0))
    c.rejects('非法alpha拒绝',lambda:LoRALinear(nn.Linear(3,3),alpha=float('nan')))
    # 解析型计算：不能把alpha/rank写反，或改变A/B矩阵的相乘顺序。
    linear=nn.Linear(3,2,bias=False).double()
    lora=LoRALinear(linear,rank=1,alpha=4).double()
    with torch.no_grad():
        lora.A.copy_(torch.tensor([[1.,2.,3.]],dtype=torch.float64))
        lora.B.copy_(torch.tensor([[.5],[-1.]],dtype=torch.float64))
    inp=torch.tensor([[1.,0.,-1.]],dtype=torch.float64)
    manual=F.linear(inp,linear.weight)+4*F.linear(F.linear(inp,lora.A),lora.B)
    c.close('LoRA解析式alpha除rank乘BA',lora(inp),manual,atol=0,rtol=0)
    c.rejects('左补齐拒绝',lambda:base(batch['input_ids'],
                                      torch.cat([torch.zeros(2,1,dtype=torch.bool),batch['attention_mask'][:,:-1]],1)))
    altered=batch['input_ids'].clone();altered[:,-1]=tok.byte_offset+ord('1')
    with torch.no_grad():future=base(altered,batch['attention_mask'])
    c.close('改变未来token不影响更早logits',future[:,:-1],before[:,:-1],atol=0,rtol=0)
    logits=torch.zeros(2,5,4,dtype=torch.float64,requires_grad=True)
    labels=torch.tensor([[IGNORE,IGNORE,2,3,IGNORE],[IGNORE,1,2,3,0]])
    loss,parts=supervised_token_loss(logits,labels)
    c.eq('只监督6个目标',parts['target_tokens'],6)
    c.close('均匀分布的正确均值',loss.detach(),math.log(4))
    loss.backward()
    c.close('忽略prompt的logits没有梯度',logits.grad[0,0],torch.zeros(4))
    c.ok('第一个被监督token由前一个位置预测',logits.grad[0,1,2]<0)
    c.close('最后位置logits没有下一个目标',logits.grad[:,-1],torch.zeros(2,4))
    c.rejects('全忽略目标拒绝',lambda:supervised_token_loss(logits.detach(),torch.full_like(labels,IGNORE)))
    c.rejects('非long标签拒绝',lambda:supervised_token_loss(logits.detach(),labels.float()))
    c.rejects('越界监督目标拒绝',lambda:supervised_token_loss(torch.zeros(1,3,4),torch.tensor([[IGNORE,4,3]])))
    badlogits=logits.detach().clone();badlogits[0,0,0]=float('nan')
    c.rejects('非有限logits拒绝',lambda:supervised_token_loss(badlogits,labels))
    # 非均匀logits防止提前移位与两次移位的假阳性。
    nonuniform=torch.tensor([[[5.,0.,0.],[0.,4.,1.],[1.,0.,6.]]],dtype=torch.float64)
    labs=torch.tensor([[IGNORE,0,1]])
    expected=(-F.log_softmax(nonuniform[0,0],-1)[0]
              -F.log_softmax(nonuniform[0,1],-1)[1])/2
    c.close('不均匀shift一次核对',supervised_token_loss(nonuniform,labs)[0],expected)
    x=torch.randn(1,4,3,dtype=torch.float64,requires_grad=True)
    y=torch.tensor([[IGNORE,IGNORE,0,1]])
    c.ok('损失梯度数值核对',torch.autograd.gradcheck(lambda a:supervised_token_loss(a,y)[0],(x,)))
    # LoRA第一步A梯度是零，B梯度非零；第二步A开始接通。全部基座冻结。
    opt=torch.optim.AdamW([p for _,p in adapters(policy)],lr=.01)
    same_data=[data['train'][0],data['train'][1]]
    before_adapter=state_hash(adapter_snapshot(policy))
    first=toy_adapter_step(policy,opt,same_data,tok)
    c.ok('首步A梯度为零',abs(first['A_grad_sum'])<1e-10)
    c.ok('首步B收到非零梯度',first['B_grad_sum']>0)
    c.ok('参数在第一步改变',state_hash(adapter_snapshot(policy))!=before_adapter)
    second=toy_adapter_step(policy,opt,same_data,tok)
    c.ok('第二步A收到梯度',second['A_grad_sum']>0)
    c.eq('两次更新后基座不变',base_hash(policy),ref_hash)
    c.eq('参考自身不变',state_hash(base.state_dict()),ref_hash)
    c.ok('参考没有梯度',all(p.grad is None for p in base.parameters()))
    bad_opt=torch.optim.AdamW(policy.parameters(),lr=.01)
    c.rejects('优化器包含冻结基座拒绝',lambda:toy_adapter_step(policy,bad_opt,same_data,tok))
    c.rejects('训练不能接受test数据',lambda:toy_adapter_step(policy,opt,data['test'][:1],tok))
    c.ok('梯度范数有限',math.isfinite(second['grad_norm_pre_clip']))
    snap=adapter_snapshot(policy);snap_hash=state_hash(snap)
    toy_adapter_step(policy,opt,same_data,tok)
    c.eq('adapter快照不会随新step变化',state_hash(snap),snap_hash)
    restore_adapter(policy,snap)
    c.eq('恢复adapter参数逐位一致',state_hash(adapter_snapshot(policy)),snap_hash)
    malformed=copy.deepcopy(snap);name=next(iter(malformed));malformed[name]=torch.ones(1)
    c.rejects('adapter形状不兼容拒绝',lambda:restore_adapter(policy,malformed))
    malformed=copy.deepcopy(snap);malformed.pop(name)
    c.rejects('adapter键缺失拒绝',lambda:restore_adapter(policy,malformed))
    report=evaluate_nll(policy,data['dev'][:2],tok)
    c.eq('评估分母两条',report['examples'],2)
    c.ok('teacher forced NLL必须有限',math.isfinite(report['teacher_forced_nll']))
    c.ok('不得把NLL当生成准确率',report['not_generation_accuracy'])
    # 静态预算不是峰值；CPU必须明确没有CUDA实测。
    accounting=memory_accounting(policy,8,100)
    c.eq('静态核算四项之和',accounting['bookkept_static_bytes'],
         sum(accounting[k] for k in ('frozen_parameter_bytes','trainable_parameter_bytes',
                                       'adapter_gradient_bytes','estimated_adam_moment_bytes')))
    c.eq('Adam状态按两倍训练参数核算',accounting['estimated_adam_moment_bytes'],
         2*accounting['trainable_parameter_bytes'])
    c.ok('T翻倍注意力示意项四倍',memory_accounting(policy,8,200)['attention_example_bytes']==
          4*memory_accounting(policy,8,100)['attention_example_bytes'])
    c.ok('示意项没有冒称GPU峰值','NOT measured peak' in accounting['warning'])
    c.eq('CPU明确无CUDA测量',cuda_metrics(torch.device('cpu'))['peak_allocated_mib'],None)
    c.rejects('无效训练预算拒绝',lambda:ToyTrainConfig(static_budget_mib=-1).validate())
    c.rejects('负cuda峰值预算拒绝',lambda:ToyTrainConfig(cuda_peak_budget_mib=-1).validate())
    c.rejects('负step数拒绝',lambda:ToyTrainConfig(steps=0).validate())
    c.rejects('非法设备拒绝',lambda:device_from_name('fpga'))
    # 磁盘恢复必须精确基座，且不修改冻结的源文件。
    with tempfile.TemporaryDirectory() as tmp:
        folder=Path(tmp)/'bundle';base.eval();policy.eval()
        fake_report={'test':'local_fixture'}
        save_toy_bundle(folder,base,policy,tok,data,ToyTrainConfig(rank=2,alpha=4),fake_report)
        reloaded_base,reloaded_policy,new_tok=load_toy_bundle(folder)
        c.eq('重载基座参数身份',state_hash(reloaded_base.state_dict()),ref_hash)
        c.eq('重载adapter身份',state_hash(adapter_snapshot(reloaded_policy)),snap_hash)
        c.ok('推理重载不允许训练',all(not p.requires_grad for p in reloaded_policy.parameters()))
        c.ok('重载词表契约一致',new_tok.contract()==tok.contract())
        with torch.no_grad():
            result1=policy(batch['input_ids'],batch['attention_mask'])
            result2=reloaded_policy(batch['input_ids'],batch['attention_mask'])
        c.close('独立磁盘重载logits一致',result1,result2,atol=0,rtol=0)
        adapter_blob=torch.load(folder/'toy_adapter.pt',weights_only=True)
        adapter_blob['base_hash']='wrong'
        torch.save(adapter_blob,folder/'toy_adapter.pt')
        c.rejects('同架构错误基座拒绝',lambda:load_toy_bundle(folder))
    # 静态本地安全门：不下载、不加载、模型目录只读检查。
    with tempfile.TemporaryDirectory() as tmp:
        model_path=Path(tmp)/'mock_hf';fake_hf_folder(model_path)
        local=inspect_local_training_candidate(model_path,max_weight_bytes=1024)
        c.eq('静态只读有一份权重',local['weight_file_count'],1)
        c.ok('静态不声称模型已训练',local['preflight']=='metadata_only_does_not_load_or_train')
        c.ok('默认非完整权重哈希',not local['cryptographic_weight_identity'])
        c.rejects('权重字节闸门拒绝',lambda:inspect_local_training_candidate(model_path,max_weight_bytes=4))
        c.rejects('输出不能写进模型目录',lambda:validate_output_outside_model(model_path,model_path/'outputs'))
        c.rejects('路径不接受仓库ID',lambda:inspect_local_training_candidate('fake/org'))
        (model_path/'tokenizer.json').unlink()
        c.rejects('缺少tokenizer候选拒绝',lambda:inspect_local_training_candidate(model_path))
        (model_path/'tokenizer.json').write_text('{}')
        with tempfile.TemporaryDirectory() as other:
            sym=Path(other)/'linked.safetensors'
            try:
                sym.symlink_to(model_path/'model.safetensors')
            except (OSError,NotImplementedError):pass
            else:
                (model_path/'model.safetensors').unlink()
                (model_path/'model.safetensors').symlink_to(sym)
                c.rejects('拒绝符号链接权重',lambda:inspect_local_training_candidate(model_path))
        # HF/PEFT不存在时，不导入、不假装实际训练。
        import importlib.util
        if importlib.util.find_spec('peft') is None:
            c.rejects('缺少PEFT明确拒绝而不下载',
                      lambda:run_local_peft_smoke(model_path,Path(tmp)/'out'))
    return {'passed':len(c.names),'names':c.names,'scope':
            'offline synthetic tokenizer+actual torch LoRA; local PEFT path untested without deps'}

In [25]:
checks=run_checks()
print('固定检查通过：',checks['passed'],'项')
print('范围：',checks['scope'])
if RUN_DIR is not None:
    write_json(RUN_DIR/'checks.json',checks)

固定检查通过： 92 项
范围： offline synthetic tokenizer+actual torch LoRA; local PEFT path untested without deps


## 14. 独立练习（默认不运行）

三道题都需要独立实现，检查器会比对行为与梯度。先补全，再将对应 `RUN_EXERCISE_*` 改为 `True`。

**练习 A：训练序列和 loss mask。** 给定 toy tokenizer 和合成一条记录，构造 assistant generation prompt 与完整 assistant 消息，检查精确前缀，再把提示全部标为 -100。EOS 必须保留，超过上限必须拒绝，而不是截断。

In [26]:
def check_exercise_a(fn):
    c=Checks();tok=ByteMergeChatTokenizer();data=make_dataset();row=data['train'][0]
    got=fn(tok,row,160);gold=encode_training_row(tok,row,160)
    c.eq('A_输入IDs一致',got['input_ids'],gold['input_ids'])
    c.eq('A_标签前缀忽略',got['labels'][:gold['prompt_len']],[IGNORE]*gold['prompt_len'])
    c.eq('A_回答及EOS一致',got['labels'][gold['prompt_len']:],gold['labels'][gold['prompt_len']:])
    c.rejects('A_超长不裁剪',lambda:fn(tok,row,8))
    return {'passed':len(c.names),'checks':c.names}


def check_exercise_b(fn):
    c=Checks();logits=torch.randn(2,5,7,dtype=torch.float64,requires_grad=True)
    labels=torch.tensor([[IGNORE,IGNORE,2,3,IGNORE],[IGNORE,1,2,3,4]])
    value=fn(logits,labels)
    gold=supervised_token_loss(logits,labels)[0]
    c.close('B_有效token平均NLL',value,gold)
    g1=torch.autograd.grad(value,logits,retain_graph=True)[0]
    g2=torch.autograd.grad(gold,logits)[0]
    c.close('B_全部logits梯度',g1,g2)
    return {'passed':len(c.names),'checks':c.names}


def check_exercise_c(fn):
    c=Checks();seed_all(6930);tok=ByteMergeChatTokenizer()
    model=TinyChatLM(ToyConfig(tok.vocab_size,dim=16,heads=2,layers=1,ff_dim=32))
    policy=inject_lora(model,rank=2,alpha=4)
    params=fn(policy)
    expected=[p for _,p in adapters(policy)]
    c.ok('C_每个返回参数都是LoRA',len(params)==len(expected) and
         {id(p) for p in params}=={id(p) for p in expected})
    c.ok('C_没有误解冻基座',all(p.requires_grad==(n.endswith(('.A','.B')))
                              for n,p in policy.named_parameters()))
    optimizer=torch.optim.SGD(params,lr=.01)
    result=toy_adapter_step(policy,optimizer,make_dataset()['train'][:2],tok)
    c.ok('C_真正能优化LoRA',result['B_grad_sum']>0)
    return {'passed':len(c.names),'checks':c.names}

In [27]:
def encode_row_ex(tok,row,max_length):
    raise NotImplementedError('请独立构造Chat Template训练边界与标签')

RUN_EXERCISE_A=False
if RUN_EXERCISE_A:
    print(check_exercise_a(encode_row_ex))

**练习 B：移位一次的 token 平均 NLL。** 输入 logits `[B,T,V]` 和同位 labels `[B,T]`，只在loss内部移位，忽略 -100，对全部有效目标求均值，保留梯度。不要调用 `.item()`。

In [28]:
def masked_next_token_nll_ex(logits,labels):
    raise NotImplementedError('请独立实现一次shift和有效token均值')

RUN_EXERCISE_B=False
if RUN_EXERCISE_B:
    print(check_exercise_b(masked_next_token_nll_ex))

**练习 C：只选择 Q/V LoRA A/B。** 把基座参数全部冻结，只让 adapter A/B 可训练，返回优化器应该接收的参数列表；不能把 embedding、LayerNorm、LM head、原始 Q/V 权重放进去。检查器会执行一次真实梯度更新。

In [29]:
def collect_lora_only_ex(policy):
    raise NotImplementedError('请只开放并返回LoRA A/B')

RUN_EXERCISE_C=False
if RUN_EXERCISE_C:
    print(check_exercise_c(collect_lora_only_ex))

## 15. 完整参考答案

先完成自己实现再核对。答案使用独立名称，不覆盖练习函数；真实工程中的更多身份检查仍应保留。

In [30]:
def answer_a(tok,row,max_length):
    prefix=tok.apply_chat_template(messages_for_row(row),tokenize=True,add_generation_prompt=True)
    full=tok.apply_chat_template(messages_for_row(row)+[
        {'role':'assistant','content':row['assistant']}],tokenize=True,add_generation_prompt=False)
    require(full[:len(prefix)]==prefix,'助手prefix与训练full不一致')
    require(full[-1]==tok.eos_token_id and len(full)<=max_length,'EOS/长度有误')
    return {'input_ids':full,'labels':[IGNORE]*len(prefix)+full[len(prefix):],
            'prompt_len':len(prefix)}


def answer_b(logits,labels):
    targets=labels[:,1:].reshape(-1)
    pred=logits[:,:-1,:].reshape(-1,logits.shape[-1])
    return F.cross_entropy(pred,targets,ignore_index=IGNORE,reduction='mean')


def answer_c(policy):
    policy.requires_grad_(False)
    selected=[]
    for module in policy.modules():
        if isinstance(module,LoRALinear):
            module.A.requires_grad_(True);module.B.requires_grad_(True)
            selected.extend([module.A,module.B])
    return selected

In [31]:
print('A:',check_exercise_a(answer_a))
print('B:',check_exercise_b(answer_b))
print('C:',check_exercise_c(answer_c))

A: {'passed': 4, 'checks': ['A_输入IDs一致', 'A_标签前缀忽略', 'A_回答及EOS一致', 'A_超长不裁剪']}
B: {'passed': 2, 'checks': ['B_有效token平均NLL', 'B_全部logits梯度']}
C: {'passed': 3, 'checks': ['C_每个返回参数都是LoRA', 'C_没有误解冻基座', 'C_真正能优化LoRA']}


## 16. CLI与实际工程边界

同一个独立 `.py` 脚本可在命令行运行：

```bash
python lesson_69_lora_preflight_data_freeze_gradient_memory.py --checks-only
python lesson_69_lora_preflight_data_freeze_gradient_memory.py
python lesson_69_lora_preflight_data_freeze_gradient_memory.py --reload-run lesson_69_outputs/某次run
```

只读静态检查你**受信任的本地文件**：

```bash
python lesson_69_lora_preflight_data_freeze_gradient_memory.py --inspect-local /绝对路径/你的本地模型
```

用户**明确启用、安装相关依赖、确认文件可信且足够小**后才可单步 PEFT：

```bash
python lesson_69_lora_preflight_data_freeze_gradient_memory.py --local-peft-smoke /绝对路径/你的本地模型 --output-dir 另一个输出目录
```

可选 `--device cuda`、`--max-weight-bytes`、`--target-modules q_proj,v_proj` 用于改变审计条件；不保证所有模型都采用这些层名。不添加自动下载或隐式回退。

In [32]:
def main(argv=None) -> int:
    parser=argparse.ArgumentParser(description=__doc__)
    mode=parser.add_mutually_exclusive_group()
    mode.add_argument('--checks-only',action='store_true')
    mode.add_argument('--inspect-local',metavar='MODEL_DIR')
    mode.add_argument('--local-peft-smoke',metavar='MODEL_DIR')
    mode.add_argument('--reload-run',metavar='TOY_DIR')
    parser.add_argument('--output-dir',default='lesson_69_outputs')
    parser.add_argument('--device',choices=['cpu','cuda','auto'],default='cpu')
    parser.add_argument('--steps',type=int,default=16)
    parser.add_argument('--max-length',type=int,default=160)
    parser.add_argument('--static-budget-mib',type=float,default=64.0)
    parser.add_argument('--cuda-peak-budget-mib',type=float,default=0.0)
    parser.add_argument('--max-weight-bytes',type=int,default=256*2**20)
    parser.add_argument('--full-weight-hash',action='store_true')
    parser.add_argument('--target-modules',default='q_proj,v_proj')
    args=parser.parse_args(argv)
    torch.set_num_threads(1)
    if args.inspect_local:
        print(json.dumps(inspect_local_training_candidate(args.inspect_local,
                 max_weight_bytes=args.max_weight_bytes,full_weight_hash=args.full_weight_hash),
                 ensure_ascii=False,indent=2));return 0
    if args.local_peft_smoke:
        target_names=tuple(x.strip() for x in args.target_modules.split(','))
        folder,report=run_local_peft_smoke(args.local_peft_smoke,args.output_dir,
               device_name=args.device,max_weight_bytes=args.max_weight_bytes,
               target_modules=target_names,max_length=args.max_length)
        print('显式本地PEFT smoke输出:',folder)
        print('仅做一个本地模型反向step，未评估泛化；模型目录未修改')
        return 0
    if args.reload_run:
        print(json.dumps(reload_toy_run(Path(args.reload_run),args.device),
              ensure_ascii=False,indent=2));return 0
    checks=run_checks()
    print('固定检查通过:',checks['passed'],'项')
    if args.checks_only:return 0
    cfg=ToyTrainConfig(steps=args.steps,max_length=args.max_length,
        static_budget_mib=args.static_budget_mib,cuda_peak_budget_mib=args.cuda_peak_budget_mib)
    folder,_=run_offline_preflight(args.output_dir,args.device,cfg)
    write_json(folder/'checks.json',checks)
    return 0

## 17. 验收与下一课

你应当能够明确回答：

- 训练和生成各自怎样构造 Chat Template，为什么比较完整 token 前缀？
- 什么时候 BOS/系统/user/assistant header 应该 `-100`，什么时候 EOS 要参与 loss？
- 为什么首步 A 梯度是零、B 梯度非零，随后 A 才开始收到梯度？
- `requires_grad=False`、优化器参数组与“内容哈希不变”各在证明什么？
- 为什么同架构同 rank 的 adapter 也不能随意换基座？
- `max_memory_allocated`、`max_memory_reserved`、静态参数字节核算分别衡量什么，为什么不能提前保证不 OOM？
- 为什么只读静态审计、toy真实PyTorch训练、可选HF+PEFT单步不能写成相同的“实测能力”？

**第 70 课预告：真实本地 PEFT 适配器的最小训练闭环与回滚——在明确预算、可信检查点和验证通过的前提下，研究 adapter-only 保存、重载、前后指标对比和失败恢复。**

### 资料

- [Hugging Face：Chat Template](https://huggingface.co/docs/transformers/main/chat_templating)
- [PEFT：Quicktour与get_peft_model](https://huggingface.co/docs/peft/quicktour)
- [PEFT：LoRA层、rank、初始化](https://huggingface.co/docs/peft/main/package_reference/lora)
- [Transformers：离线与local_files_only](https://huggingface.co/docs/transformers/installation)
- [PyTorch：CUDA allocated/reserved峰值](https://docs.pytorch.org/docs/stable/generated/torch.cuda.memory.max_memory_allocated.html)
- [Safetensors：格式与加载边界](https://huggingface.co/docs/safetensors/index)

本课所有默认代码在CPU运行。没有可用GPU、预训练HF权重或本地PEFT依赖时，绝不编造相应实测结果。